# DKD Risk Prediction in Type 2 Diabetes
## 02. Cohort Construction and Feature Curation

This notebook constructs landmark-specific DKD prediction cohorts from the curated outputs generated by `01_data_curation_20260821_FINAL.ipynb`.

The workflow:

1. Loads the curated participant-level and longitudinal phenotype datasets.
2. Constructs the T2D source cohort.
3. Applies landmark-specific risk-set eligibility.
4. Assigns future incident DKD cases and eligible controls.
5. Constructs predictor features using information available strictly before each landmark.
6. Saves validated landmark cohorts and modeling-ready feature matrices.

No raw eGFR or UACR derivation is repeated in this notebook.


# Setup


In [ ]:
import json
import os
import re
import subprocess
import warnings
import gc

import gcsfs
import numpy as np
import pandas as pd
from google.cloud import bigquery

warnings.filterwarnings(
    "ignore",
    message="Unable to represent RANGE schema as struct using pandas ArrowDtype",
)

# ============================================================
# Migrated All of Us workspace
# ============================================================

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

DATA_CURATED_DIR = f"{bucket}/aou_dkd/dkd_data_curation_20260821"
CURATED_DIR = f"{bucket}/aou_dkd/dkd_cohort_20260821"

fs = gcsfs.GCSFileSystem()


# ============================================================
# Resolve the All of Us CDR BigQuery resource
# ============================================================

def resolve_aou_cdr():
    """
    Resolve the Controlled Tier All of Us CDR attached to the
    current Verily Workbench workspace.
    """
    result = subprocess.run(
        ["wb", "resource", "list", "--type=BQ_DATASET", "--format=JSON"],
        capture_output=True,
        text=True,
        check=True,
    )

    resources = json.loads(result.stdout)
    candidates = []

    for resource in resources:
        dataset_id = str(resource.get("datasetId", ""))

        if re.fullmatch(r"C\d{4}Q\d+R\d+", dataset_id):
            project_id = resource.get("projectId")

            if project_id:
                candidates.append(
                    {
                        "resource_id": resource.get("id"),
                        "path": f"{project_id}.{dataset_id}",
                    }
                )

    if len(candidates) == 0:
        raise RuntimeError(
            "No All of Us Controlled Tier CDR BigQuery resource was found."
        )

    if len(candidates) > 1:
        available = "\n".join(
            f"  {x['resource_id']}: {x['path']}"
            for x in candidates
        )
        raise RuntimeError(
            "Multiple All of Us Controlled Tier CDR resources were found.\n"
            "Set CDR_DATASET explicitly to the intended project.dataset.\n"
            f"{available}"
        )

    return candidates[0]["path"]


CDR_DATASET = resolve_aou_cdr()

job_query_config = bigquery.QueryJobConfig(
    default_dataset=CDR_DATASET
)

bq_client = bigquery.Client(
    default_query_job_config=job_query_config
)

print("Workspace:", bucket)
print("Curated input directory:", DATA_CURATED_DIR)
print("Cohort output directory:", CURATED_DIR)
print("All of Us CDR:", CDR_DATASET)


## Data I/O

Notebook 02 reads only stable outputs from the data-curation notebook and writes all cohort and feature outputs to the dated cohort directory.


In [ ]:
def read_gcs_csv(path, **kwargs):
    """Read a CSV directly from Google Cloud Storage."""
    with fs.open(path, "rb") as f:
        return pd.read_csv(f, **kwargs)


def read_gcs_parquet(path, **kwargs):
    """Read a Parquet file directly from Google Cloud Storage."""
    with fs.open(path, "rb") as f:
        return pd.read_parquet(f, **kwargs)


def save_gcs_csv(df, path, **kwargs):
    """Save a DataFrame as CSV to Google Cloud Storage."""
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False, **kwargs)

    print(f"Saved CSV: {path} | shape={df.shape}")


def save_gcs_parquet(df, path, **kwargs):
    """Save a DataFrame as Parquet to Google Cloud Storage."""
    with fs.open(path, "wb") as f:
        df.to_parquet(f, index=False, **kwargs)

    print(f"Saved Parquet: {path} | shape={df.shape}")


def save_gcs_pickle(df, path):
    """Save a pandas pickle directly to Google Cloud Storage."""
    with fs.open(path, "wb") as f:
        df.to_pickle(f)

    print(f"Saved pickle: {path} | shape={df.shape}")


# Compatibility wrapper for feature-building code.
def save_phenotypes(df, bucket, filename, **kwargs):
    path = f"{CURATED_DIR}/{filename}"
    save_gcs_parquet(
        df,
        re.sub(r"\.csv$", ".parquet", path),
    )
    return path


## Common Cleaning Functions


In [ ]:
BAD_DATE_TOKENS = {
    "",
    "null",
    "none",
    "nan",
    "na",
    "n/a",
    "nat",
    "0000-00-00",
    "0000-00-00 00:00:00",
}


def parse_datetime(series):
    """Parse a date-like series as UTC datetime."""
    s = pd.Series(series)

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s, errors="coerce", utc=True)

    s = s.astype("string").str.strip()
    s = s.where(~s.str.lower().isin(BAD_DATE_TOKENS), pd.NA)

    parsed = pd.to_datetime(s, errors="coerce", utc=True)

    missing = parsed.isna() & s.notna()

    if missing.any():
        ymd = s.loc[missing].str.extract(
            r"(\d{4}-\d{2}-\d{2})",
            expand=False,
        )
        parsed.loc[missing] = pd.to_datetime(
            ymd,
            errors="coerce",
            utc=True,
        )

    return parsed


def normalize_person_id(df, label="data"):
    """Standardize person_id as nullable Int64 and remove invalid IDs."""
    if "person_id" not in df.columns:
        raise KeyError(f"{label}: missing person_id")

    out = df.copy()

    out["person_id"] = pd.to_numeric(
        out["person_id"],
        errors="coerce",
    ).astype("Int64")

    n_invalid = int(out["person_id"].isna().sum())

    if n_invalid:
        print(
            f"{label}: removing {n_invalid:,} rows "
            "with invalid person_id"
        )
        out = out.dropna(subset=["person_id"]).copy()

    return out


def earliest_per_person(df, date_col, out_col, label):
    """Return the earliest valid event date for each participant."""
    if date_col not in df.columns:
        raise KeyError(
            f"{label}: missing date column '{date_col}'"
        )

    x = normalize_person_id(
        df[["person_id", date_col]],
        label=label,
    )

    x[date_col] = parse_datetime(x[date_col])

    return (
        x.dropna(subset=[date_col])
        .groupby("person_id", as_index=False)[date_col]
        .min()
        .rename(columns={date_col: out_col})
    )


def ensure_person_level(df, name):
    """Require one row per participant before feature-block merges."""
    if "person_id" not in df.columns:
        raise ValueError(f"{name}: missing person_id")

    if df["person_id"].duplicated().any():
        n_dup = int(df["person_id"].duplicated().sum())
        raise ValueError(
            f"{name}: {n_dup} duplicate person_id rows detected"
        )


# 1. Load Curated Inputs

The data-curation notebook provides the participant-level phenotype dates and reusable longitudinal renal measurement tables required for cohort construction.


In [ ]:
INPUT_PATHS = {
    "demographics": (
        f"{DATA_CURATED_DIR}/participant_demographics.parquet"
    ),
    "ehr": (
        f"{DATA_CURATED_DIR}/derived_ehr_dates.csv"
    ),
    "t2d": (
        f"{DATA_CURATED_DIR}/final_t2d_first_evidence.csv"
    ),
    "ckd_icd": (
        f"{DATA_CURATED_DIR}/ckd_dignosis_icd_codes.csv"
    ),
    "dkd_icd": (
        f"{DATA_CURATED_DIR}/dkd_dignosis_icd_codes.csv"
    ),
    "uacr_onset": (
        f"{DATA_CURATED_DIR}/person_with_ckd_any_uacr.csv"
    ),
    "egfr_onset": (
        f"{DATA_CURATED_DIR}/person_with_ckd_any_egfr.csv"
    ),
    "dialysis_codes": (
        f"{DATA_CURATED_DIR}/ckd_dignosis_dialysis_codes.csv"
    ),
    "dialysis_labs": (
        f"{DATA_CURATED_DIR}/ckd_dignosis_dialysis_labs.csv"
    ),
    "transplant": (
        f"{DATA_CURATED_DIR}/ckd_dignosis_transplant_codes.csv"
    ),
    "uacr_long": (
        f"{DATA_CURATED_DIR}/longitudinal_uacr_measurements.parquet"
    ),
    "egfr_long": (
        f"{DATA_CURATED_DIR}/longitudinal_egfr_measurements.parquet"
    ),
    "computed_egfr": (
        f"{DATA_CURATED_DIR}/computed_egfr_measurements.parquet"
    ),
    "serum_creatinine": (
        f"{DATA_CURATED_DIR}/longitudinal_serum_creatinine.parquet"
    ),
    "spot_urine_albumin": (
        f"{DATA_CURATED_DIR}/longitudinal_spot_urine_albumin.parquet"
    ),
    "a1c": (
        f"{DATA_CURATED_DIR}/aou_a1c_labs.csv"
    ),
    "fbg": (
        f"{DATA_CURATED_DIR}/aou_fbg_labs.csv"
    ),
    "rbg": (
        f"{DATA_CURATED_DIR}/all_random_blood_glucose.parquet"
    ),
    "basic_surveys": (
        f"{DATA_CURATED_DIR}/aou_basic_surveys.csv"
    ),
    "social_surveys": (
        f"{DATA_CURATED_DIR}/aou_social_health_surveys.csv"
    ),
    "lifestyle_surveys": (
        f"{DATA_CURATED_DIR}/aou_lifestyle_surveys.csv"
    ),
    "family_history_surveys": (
        f"{DATA_CURATED_DIR}/aou_personal_family_history_surveys.csv"
    ),
}


missing_inputs = [
    path
    for path in INPUT_PATHS.values()
    if not fs.exists(path)
]

if missing_inputs:
    raise FileNotFoundError(
        "Notebook 01 has not produced all required curated inputs:\n"
        + "\n".join(missing_inputs)
    )


In [ ]:
participant_demographics = read_gcs_parquet(
    INPUT_PATHS["demographics"]
)

ehr_dates = read_gcs_csv(INPUT_PATHS["ehr"])
final_t2d_dx = read_gcs_csv(INPUT_PATHS["t2d"])

ckd_icd_earliest = read_gcs_csv(INPUT_PATHS["ckd_icd"])
dkd_icd_earliest = read_gcs_csv(INPUT_PATHS["dkd_icd"])

either_uacr = read_gcs_csv(INPUT_PATHS["uacr_onset"])
ckd_egfr_any = read_gcs_csv(INPUT_PATHS["egfr_onset"])

dialysis_code_clean_earliest = read_gcs_csv(
    INPUT_PATHS["dialysis_codes"]
)
first_dialysis_lab = read_gcs_csv(
    INPUT_PATHS["dialysis_labs"]
)
ckd_replac_earliest = read_gcs_csv(
    INPUT_PATHS["transplant"]
)

df_uacr_long = read_gcs_parquet(
    INPUT_PATHS["uacr_long"]
)
df_egfr_long = read_gcs_parquet(
    INPUT_PATHS["egfr_long"]
)
df_computed_egfr = read_gcs_parquet(
    INPUT_PATHS["computed_egfr"]
)
df_scr_long = read_gcs_parquet(
    INPUT_PATHS["serum_creatinine"]
)
df_ualb_long = read_gcs_parquet(
    INPUT_PATHS["spot_urine_albumin"]
)

print(f"T2D participants: {final_t2d_dx['person_id'].nunique():,}")
print(f"Longitudinal eGFR rows: {len(df_egfr_long):,}")
print(f"Computed eGFR rows: {len(df_computed_egfr):,}")
print(f"Longitudinal UACR rows: {len(df_uacr_long):,}")


# 2. T2D Source Cohort

The source cohort is anchored at the first confirmed T2D date.

Participants must:

- Have a valid T2D index date and EHR observation window.
- Have at least 365 days of observable EHR history before T2D diagnosis.
- Be at least 18 years old at T2D diagnosis.
- Have no qualifying DKD evidence on or before the T2D index date.

Kidney disease occurring after T2D is retained at this stage. The 180-day incident-DKD rule and landmark-specific risk-set rules are applied separately below.


In [ ]:
# ============================================================
# Prespecified cohort definitions
# ============================================================

WASHOUT_DAYS = 365
ADULT_AGE_YEARS = 18

# A qualifying kidney event must occur at least 180 days after
# T2D diagnosis to be considered incident DKD.
INCIDENT_LAG_DAYS = 180

# Preserve the original control-ascertainment requirements.
MIN_CONTROL_FOLLOWUP_DAYS = 730
MIN_CONTROL_COMPUTED_EGFR_MEASUREMENTS = 2


## Harmonized Kidney Event Date

The first qualifying kidney event is the earliest date from:

- Persistent reduced eGFR.
- Persistent albuminuria.
- DKD-specific diagnosis codes.
- Dialysis.
- Kidney transplantation.

General CKD diagnosis codes are retained as an audit flag but do not independently define the DKD event.


In [ ]:
# Standardize participant-level source tables.
t2d = normalize_person_id(final_t2d_dx, "T2D")
ehr = normalize_person_id(ehr_dates, "EHR dates")

ckd_icd = normalize_person_id(
    ckd_icd_earliest,
    "CKD ICD",
)
dkd_icd = normalize_person_id(
    dkd_icd_earliest,
    "DKD ICD",
)
egfr = normalize_person_id(
    ckd_egfr_any,
    "Persistent eGFR",
)
uacr = normalize_person_id(
    either_uacr,
    "Persistent UACR",
)
dialysis_codes = normalize_person_id(
    dialysis_code_clean_earliest,
    "Dialysis codes",
)
dialysis_labs = normalize_person_id(
    first_dialysis_lab,
    "Dialysis labs",
)
transplant = normalize_person_id(
    ckd_replac_earliest,
    "Kidney transplant",
)

t2d["t2d_first_evidence_date"] = parse_datetime(
    t2d["t2d_first_evidence_date"]
)

ehr["ehr_start_date"] = parse_datetime(
    ehr["ehr_start_date"]
)
ehr["ehr_end_date"] = parse_datetime(
    ehr["ehr_end_date"]
)


In [ ]:
# Reduce every kidney phenotype to one dated event per participant.
egfr_onset = earliest_per_person(
    egfr,
    "ckd_egfr_onset_date",
    "egfr_onset_date",
    "Persistent eGFR",
)

uacr_onset = earliest_per_person(
    uacr,
    "first_persistent_albuminuria_any_date",
    "uacr_onset_date",
    "Persistent UACR",
)

dkd_icd_onset = earliest_per_person(
    dkd_icd,
    "condition_start_datetime",
    "dkd_icd_date",
    "DKD ICD",
)

dialysis_code_onset = earliest_per_person(
    dialysis_codes,
    "condition_start_datetime",
    "dialysis_code_date",
    "Dialysis codes",
)

dialysis_lab_onset = earliest_per_person(
    dialysis_labs,
    "measurement_datetime",
    "dialysis_lab_date",
    "Dialysis labs",
)

transplant_onset = earliest_per_person(
    transplant,
    "condition_start_datetime",
    "transplant_date",
    "Kidney transplant",
)


In [ ]:
# Build the participant-level T2D/EHR backbone.
t2d_columns = ["person_id", "t2d_first_evidence_date"]

for col in [
    "t2d_first_evidence_source",
    "t2d_source_count",
    "classification",
]:
    if col in t2d.columns:
        t2d_columns.append(col)

ehr_core = (
    ehr[
        [
            "person_id",
            "ehr_start_date",
            "ehr_end_date",
        ]
    ]
    .sort_values("person_id")
    .drop_duplicates("person_id")
)

base = (
    t2d[t2d_columns]
    .sort_values("person_id")
    .drop_duplicates("person_id")
    .merge(
        ehr_core,
        on="person_id",
        how="left",
        validate="one_to_one",
    )
)

# Add date of birth here so the adult eligibility criterion is
# explicit in cohort construction rather than hidden downstream.
dob = (
    participant_demographics[
        ["person_id", "date_of_birth"]
    ]
    .drop_duplicates("person_id")
    .copy()
)

dob["person_id"] = pd.to_numeric(
    dob["person_id"],
    errors="coerce",
).astype("Int64")

dob["date_of_birth"] = parse_datetime(
    dob["date_of_birth"]
)

base = base.merge(
    dob,
    on="person_id",
    how="left",
    validate="one_to_one",
)

# Add all dated kidney phenotypes.
for phenotype in [
    egfr_onset,
    uacr_onset,
    dkd_icd_onset,
    dialysis_code_onset,
    dialysis_lab_onset,
    transplant_onset,
]:
    base = base.merge(
        phenotype,
        on="person_id",
        how="left",
        validate="one_to_one",
    )


In [ ]:
kidney_date_cols = [
    "egfr_onset_date",
    "uacr_onset_date",
    "dkd_icd_date",
    "dialysis_code_date",
    "dialysis_lab_date",
    "transplant_date",
]

base["kidney_first_evidence_date"] = (
    base[kidney_date_cols].min(axis=1)
)

source_labels = {
    "egfr_onset_date": "eGFR_persistent",
    "uacr_onset_date": "UACR_persistent",
    "dkd_icd_date": "DKD_ICD",
    "dialysis_code_date": "Dialysis_code",
    "dialysis_lab_date": "Dialysis_lab",
    "transplant_date": "Transplant",
}


def identify_kidney_source(row):
    """Preserve all sources occurring on the first qualifying event date."""
    event_date = row["kidney_first_evidence_date"]

    if pd.isna(event_date):
        return pd.NA

    sources = [
        source_labels[col]
        for col in kidney_date_cols
        if pd.notna(row[col])
        and row[col] == event_date
    ]

    return "|".join(sources) if sources else pd.NA


base["kidney_first_evidence_source"] = base.apply(
    identify_kidney_source,
    axis=1,
)

base["has_kidney"] = (
    base["kidney_first_evidence_date"].notna()
).astype("int8")

# General CKD ICD is retained only as an audit variable.
ckd_icd_flag = (
    ckd_icd[["person_id"]]
    .drop_duplicates()
    .assign(has_ckd_icd=1)
)

base = base.merge(
    ckd_icd_flag,
    on="person_id",
    how="left",
    validate="one_to_one",
)

base["has_ckd_icd"] = (
    base["has_ckd_icd"]
    .fillna(0)
    .astype("int8")
)


## Source Cohort Eligibility

The source cohort establishes the population eligible to enter the incident-DKD framework. It does not yet assign cases or controls.

A one-year pre-T2D washout is required to reduce inclusion of pre-existing kidney disease. Participants with qualifying DKD on or before T2D diagnosis are excluded.


In [ ]:
# Valid EHR observation must span the T2D index date.
base["valid_ehr_window"] = (
    base["t2d_first_evidence_date"].notna()
    & base["ehr_start_date"].notna()
    & base["ehr_end_date"].notna()
    & (
        base["ehr_start_date"]
        <= base["t2d_first_evidence_date"]
    )
    & (
        base["ehr_end_date"]
        >= base["t2d_first_evidence_date"]
    )
)

# Pre-T2D observation history.
base["washout_days"] = (
    base["t2d_first_evidence_date"]
    - base["ehr_start_date"]
).dt.days

base["passes_washout"] = (
    base["washout_days"] >= WASHOUT_DAYS
)

# Adult T2D eligibility.
base["age_at_t2d"] = (
    base["t2d_first_evidence_date"]
    - base["date_of_birth"]
).dt.days / 365.25

base["adult_at_t2d"] = (
    base["age_at_t2d"]
    >= ADULT_AGE_YEARS
)

# Kidney disease present at or before T2D is prevalent at index.
base["kidney_pre_or_on_t2d"] = (
    base["kidney_first_evidence_date"].notna()
    & (
        base["kidney_first_evidence_date"]
        <= base["t2d_first_evidence_date"]
    )
)

source_cohort = base.loc[
    base["valid_ehr_window"]
    & base["passes_washout"]
    & base["adult_at_t2d"]
    & ~base["kidney_pre_or_on_t2d"]
].copy()

source_cohort = (
    source_cohort
    .sort_values("person_id")
    .reset_index(drop=True)
)

print(f"Initial T2D population:        {len(base):,}")
print(
    f"Invalid EHR window:            "
    f"{(~base['valid_ehr_window']).sum():,}"
)
print(
    f"Insufficient 365-day washout:  "
    f"{(~base['passes_washout']).sum():,}"
)
print(
    f"Age <18 or missing at T2D:     "
    f"{(~base['adult_at_t2d']).sum():,}"
)
print(
    f"Kidney disease on/before T2D:  "
    f"{base['kidney_pre_or_on_t2d'].sum():,}"
)
print(f"T2D source cohort:             {len(source_cohort):,}")


In [ ]:
# Source-cohort integrity checks.
assert source_cohort["person_id"].is_unique
assert source_cohort["t2d_first_evidence_date"].notna().all()
assert source_cohort["adult_at_t2d"].all()

assert (
    source_cohort["washout_days"]
    >= WASHOUT_DAYS
).all()

assert not source_cohort[
    "kidney_pre_or_on_t2d"
].any()

assert not (
    source_cohort["kidney_first_evidence_date"].notna()
    & (
        source_cohort["kidney_first_evidence_date"]
        > source_cohort["ehr_end_date"]
    )
).any()

print("All source-cohort QC checks passed.")

save_gcs_parquet(
    source_cohort,
    f"{CURATED_DIR}/aou_t2d_source_cohort.parquet",
)


# 3. Incident DKD and Control Ascertainment

Incident DKD and landmarking are separate temporal rules.

**Incident-DKD rule:** a qualifying kidney event must occur at least 180 days after the T2D index date. Kidney events occurring earlier than 180 days are not classified as incident DKD and are excluded from the analytic pool.

**Control rule:** controls must have no qualifying kidney event during observed follow-up, at least 730 days of EHR follow-up after T2D, and at least two post-T2D computed-eGFR measurements.

These rules define the common analytic pool from which each landmark-specific risk set is constructed.


## Renal Surveillance for Control Ascertainment

The control-surveillance rule is reproduced from the original cohort definition. The count of post-T2D renal assessments is based specifically on the creatinine-derived eGFR table, rather than on the combined reported/computed eGFR feature table.


In [ ]:
def count_measurements_after_t2d(
    source_df,
    measurement_df,
    date_col,
    out_col,
):
    """Count measurement rows occurring on or after T2D diagnosis."""
    m = normalize_person_id(
        measurement_df[
            ["person_id", date_col]
        ],
        label=out_col,
    )

    m[date_col] = parse_datetime(
        m[date_col]
    )

    dates = source_df[
        [
            "person_id",
            "t2d_first_evidence_date",
        ]
    ].copy()

    x = m.merge(
        dates,
        on="person_id",
        how="inner",
        validate="many_to_one",
    )

    x = x.loc[
        x[date_col].notna()
        & (
            x[date_col]
            >= x["t2d_first_evidence_date"]
        )
    ].copy()

    counts = (
        x.groupby("person_id")
        .size()
        .reset_index(name=out_col)
    )

    out = source_df.merge(
        counts,
        on="person_id",
        how="left",
        validate="one_to_one",
    )

    out[out_col] = (
        out[out_col]
        .fillna(0)
        .astype(int)
    )

    return out


def add_any_measurement_after_t2d(
    source_df,
    measurement_df,
    date_col,
    out_col,
):
    """Flag whether any qualifying measurement occurs on or after T2D."""
    m = normalize_person_id(
        measurement_df[
            ["person_id", date_col]
        ],
        label=out_col,
    )

    m[date_col] = parse_datetime(
        m[date_col]
    )

    dates = source_df[
        [
            "person_id",
            "t2d_first_evidence_date",
        ]
    ].copy()

    x = m.merge(
        dates,
        on="person_id",
        how="inner",
        validate="many_to_one",
    )

    x = x.loc[
        x[date_col].notna()
        & (
            x[date_col]
            >= x["t2d_first_evidence_date"]
        )
    ].copy()

    flags = (
        x[["person_id"]]
        .drop_duplicates()
        .assign(**{out_col: 1})
    )

    out = source_df.merge(
        flags,
        on="person_id",
        how="left",
        validate="one_to_one",
    )

    out[out_col] = (
        out[out_col]
        .fillna(0)
        .astype("int8")
    )

    return out


In [ ]:
# Preserve the original >=2 computed-eGFR surveillance criterion.
source_cohort = count_measurements_after_t2d(
    source_df=source_cohort,
    measurement_df=df_computed_egfr,
    date_col="measurement_datetime",
    out_col="n_egfr_after_t2d",
)

# Any renal assessment after T2D is retained as an audit variable.
source_cohort = add_any_measurement_after_t2d(
    source_df=source_cohort,
    measurement_df=df_egfr_long,
    date_col="measurement_datetime",
    out_col="has_egfr_or_creat_after_t2d",
)

# Urine surveillance is also retained for reproducibility/QC.
source_cohort = add_any_measurement_after_t2d(
    source_df=source_cohort,
    measurement_df=df_uacr_long,
    date_col="measurement_datetime",
    out_col="has_uacr_or_ualb_after_t2d",
)

print(
    "Any post-T2D eGFR/creatinine assessment:",
    f"{source_cohort['has_egfr_or_creat_after_t2d'].sum():,}",
)

print(
    "At least 2 post-T2D computed-eGFR measurements:",
    f"{(source_cohort['n_egfr_after_t2d'] >= 2).sum():,}",
)

print(
    "Any post-T2D UACR assessment:",
    f"{source_cohort['has_uacr_or_ualb_after_t2d'].sum():,}",
)


## Common Incident-DKD Analytic Pool

The 180-day lag is applied before landmark construction.

Participants with kidney disease during the first 180 days after T2D are excluded from both case and control groups. Participants with DKD at or after day 180 are eligible incident-case candidates. Participants with no qualifying kidney event may be eligible controls if the control follow-up and surveillance requirements are satisfied.


In [ ]:
# Time from T2D to the first qualifying kidney event.
source_cohort["days_kidney_after_t2d"] = (
    source_cohort["kidney_first_evidence_date"]
    - source_cohort["t2d_first_evidence_date"]
).dt.days

source_cohort["incident_eligibility_date"] = (
    source_cohort["t2d_first_evidence_date"]
    + pd.Timedelta(days=INCIDENT_LAG_DAYS)
)

# Kidney disease after T2D but before the 180-day incident threshold
# does not satisfy the study's incident-DKD definition.
source_cohort["early_kidney_event"] = (
    source_cohort["kidney_first_evidence_date"].notna()
    & (
        source_cohort["days_kidney_after_t2d"]
        < INCIDENT_LAG_DAYS
    )
)

# Global incident-case candidate, before applying any prediction landmark.
source_cohort["incident_case_candidate"] = (
    source_cohort["kidney_first_evidence_date"].notna()
    & (
        source_cohort["days_kidney_after_t2d"]
        >= INCIDENT_LAG_DAYS
    )
)

# Preserve the original urine-rule audit.
source_cohort["persistent_albuminuria_after_lag"] = (
    source_cohort["uacr_onset_date"].notna()
    & (
        source_cohort["uacr_onset_date"]
        >= source_cohort["incident_eligibility_date"]
    )
)

source_cohort["passes_urine_control_rule"] = (
    source_cohort["has_uacr_or_ualb_after_t2d"].eq(0)
    | ~source_cohort["persistent_albuminuria_after_lag"]
)

# Original EHR follow-up definition for control eligibility.
source_cohort["followup_days_from_t2d"] = (
    source_cohort["ehr_end_date"]
    - source_cohort["t2d_first_evidence_date"]
).dt.days

source_cohort["eligible_control"] = (
    source_cohort["kidney_first_evidence_date"].isna()
    & (
        source_cohort["followup_days_from_t2d"]
        >= MIN_CONTROL_FOLLOWUP_DAYS
    )
    & source_cohort["passes_urine_control_rule"]
    & source_cohort["has_egfr_or_creat_after_t2d"].eq(1)
    & (
        source_cohort["n_egfr_after_t2d"]
        >= MIN_CONTROL_COMPUTED_EGFR_MEASUREMENTS
    )
)

# The common analytic pool contains only valid incident-case candidates
# and eligible controls. Early kidney events are excluded entirely.
analytic_pool = source_cohort.loc[
    source_cohort["incident_case_candidate"]
    | source_cohort["eligible_control"]
].copy()

analytic_pool["dkd_binary_source"] = np.where(
    analytic_pool["incident_case_candidate"],
    1,
    0,
).astype("int8")

print("Common incident-DKD analytic pool")
print("=" * 48)
print(f"T2D source cohort:                  {len(source_cohort):,}")
print(
    f"Kidney event <180 days:             "
    f"{source_cohort['early_kidney_event'].sum():,}"
)
print(
    f"Incident case candidates (>=180d):  "
    f"{source_cohort['incident_case_candidate'].sum():,}"
)
print(
    f"Eligible controls:                  "
    f"{source_cohort['eligible_control'].sum():,}"
)
print(f"Final common analytic pool:         {len(analytic_pool):,}")


In [ ]:
# Incident/control definition QC.
assert analytic_pool["person_id"].is_unique

assert not analytic_pool[
    "early_kidney_event"
].any()

assert (
    analytic_pool.loc[
        analytic_pool["incident_case_candidate"],
        "days_kidney_after_t2d",
    ]
    >= INCIDENT_LAG_DAYS
).all()

assert analytic_pool.loc[
    analytic_pool["eligible_control"],
    "kidney_first_evidence_date",
].isna().all()

assert (
    analytic_pool.loc[
        analytic_pool["eligible_control"],
        "followup_days_from_t2d",
    ]
    >= MIN_CONTROL_FOLLOWUP_DAYS
).all()

assert (
    analytic_pool.loc[
        analytic_pool["eligible_control"],
        "n_egfr_after_t2d",
    ]
    >= MIN_CONTROL_COMPUTED_EGFR_MEASUREMENTS
).all()

print("All incident-DKD/control QC checks passed.")

save_gcs_parquet(
    analytic_pool,
    f"{CURATED_DIR}/aou_dkd_common_analytic_pool.parquet",
)


# 4. Landmark Cohort Construction

Landmarking determines when prediction occurs. It does not replace the 180-day incident-DKD definition.

Four fixed prediction landmarks are evaluated:

- 30 days after T2D diagnosis.
- 6 months after T2D diagnosis.
- 1 year after T2D diagnosis.
- 2 years after T2D diagnosis.

For each landmark:

1. The participant must remain under EHR observation at the landmark.
2. An incident-case candidate is retained only if DKD occurs strictly after the landmark.
3. An otherwise valid incident case whose DKD has already occurred on or before the landmark is excluded from that landmark risk set.
4. Eligible controls are retained if they remain observable at the landmark.

Thus, the 180-day rule defines incident DKD, while the landmark rule ensures that the outcome is still in the future when prediction occurs.


In [ ]:
LANDMARKS = {
    "30d": 30,
    "6m": 180,
    "1y": 365,
    "2y": 730,
}


In [ ]:
def build_landmark_cohort(
    analytic_df,
    landmark_days,
    landmark_name,
):
    """
    Construct one landmark-specific prediction risk set from the
    common incident-DKD analytic pool.
    """
    df = analytic_df.copy()

    for col in [
        "t2d_first_evidence_date",
        "ehr_end_date",
        "kidney_first_evidence_date",
    ]:
        df[col] = parse_datetime(df[col])

    # Fixed clinical assessment point.
    df["cutoff_date"] = (
        df["t2d_first_evidence_date"]
        + pd.to_timedelta(
            landmark_days,
            unit="D",
        )
    )

    df["landmark_name"] = landmark_name
    df["landmark_days"] = int(landmark_days)

    # Everyone must still be under observation at prediction.
    df["observable_at_landmark"] = (
        df["ehr_end_date"]
        >= df["cutoff_date"]
    )

    # This is the correction that was diagnosed but not enforced
    # in the previous workflow: an incident case that has already
    # occurred cannot remain in a later landmark risk set.
    df["incident_dkd_on_or_before_landmark"] = (
        df["incident_case_candidate"]
        & (
            df["kidney_first_evidence_date"]
            <= df["cutoff_date"]
        )
    )

    # Cases must satisfy BOTH temporal rules:
    #   1) global incident definition: DKD >= T2D + 180 days
    #   2) landmark definition: DKD strictly after prediction
    df["landmark_case"] = (
        df["incident_case_candidate"]
        & df["observable_at_landmark"]
        & (
            df["kidney_first_evidence_date"]
            > df["cutoff_date"]
        )
    )

    # Controls were already defined globally. At the landmark,
    # they only need to remain under EHR observation.
    df["landmark_control"] = (
        df["eligible_control"]
        & df["observable_at_landmark"]
    )

    cohort = df.loc[
        df["landmark_case"]
        | df["landmark_control"]
    ].copy()

    cohort["dkd_binary"] = np.where(
        cohort["landmark_case"],
        1,
        0,
    ).astype("int8")

    cohort["followup_days_after_landmark"] = (
        cohort["ehr_end_date"]
        - cohort["cutoff_date"]
    ).dt.days

    cohort["days_landmark_to_dkd"] = (
        cohort["kidney_first_evidence_date"]
        - cohort["cutoff_date"]
    ).dt.days

    cohort.loc[
        cohort["dkd_binary"].eq(0),
        "days_landmark_to_dkd",
    ] = np.nan

    cohort = (
        cohort
        .sort_values("person_id")
        .reset_index(drop=True)
    )

    print(f"\n{landmark_name}")
    print("=" * 52)
    print(f"Common analytic pool:             {len(df):,}")
    print(
        f"Not observable at landmark:       "
        f"{(~df['observable_at_landmark']).sum():,}"
    )
    print(
        f"Incident DKD already occurred:    "
        f"{df['incident_dkd_on_or_before_landmark'].sum():,}"
    )
    print(
        f"Future incident DKD cases:        "
        f"{df['landmark_case'].sum():,}"
    )
    print(
        f"Eligible controls at landmark:    "
        f"{df['landmark_control'].sum():,}"
    )
    print(f"Final landmark cohort:            {len(cohort):,}")
    print(f"Case rate:                        {cohort['dkd_binary'].mean():.3f}")

    return cohort


In [ ]:
landmark_cohorts = {
    name: build_landmark_cohort(
        analytic_df=analytic_pool,
        landmark_days=days,
        landmark_name=name,
    )
    for name, days in LANDMARKS.items()
}

# Familiar aliases retained for downstream feature-building functions.
df_cutoff_baseline = landmark_cohorts["30d"].copy()
df_cutoff_6m = landmark_cohorts["6m"].copy()
df_cutoff_1y = landmark_cohorts["1y"].copy()
df_cutoff_2y = landmark_cohorts["2y"].copy()

cutoff_baseline = df_cutoff_baseline[
    ["person_id", "cutoff_date"]
].copy()

cutoffs_6m = df_cutoff_6m[
    ["person_id", "cutoff_date"]
].copy()

cutoffs_1y = df_cutoff_1y[
    ["person_id", "cutoff_date"]
].copy()

cutoffs_2y = df_cutoff_2y[
    ["person_id", "cutoff_date"]
].copy()


## Landmark Cohort Quality Control

Each landmark cohort must satisfy both the global incident-DKD definition and the landmark-specific future-outcome definition.


In [ ]:
def validate_landmark_cohort(
    cohort,
    landmark_name,
):
    """Validate one landmark-specific prediction risk set."""
    cases = cohort["dkd_binary"].eq(1)
    controls = cohort["dkd_binary"].eq(0)

    assert cohort["person_id"].is_unique
    assert cohort["cutoff_date"].notna().all()

    # All participants are observable at prediction.
    assert (
        cohort["ehr_end_date"]
        >= cohort["cutoff_date"]
    ).all()

    # All cases satisfy the original >=180-day incident-DKD rule.
    assert (
        cohort.loc[
            cases,
            "days_kidney_after_t2d",
        ]
        >= INCIDENT_LAG_DAYS
    ).all()

    # All cases also occur strictly after the landmark.
    assert (
        cohort.loc[
            cases,
            "kidney_first_evidence_date",
        ]
        > cohort.loc[
            cases,
            "cutoff_date",
        ]
    ).all()

    # No control has any qualifying kidney event.
    assert cohort.loc[
        controls,
        "kidney_first_evidence_date",
    ].isna().all()

    # Control ascertainment rules remain unchanged.
    assert (
        cohort.loc[
            controls,
            "followup_days_from_t2d",
        ]
        >= MIN_CONTROL_FOLLOWUP_DAYS
    ).all()

    assert (
        cohort.loc[
            controls,
            "n_egfr_after_t2d",
        ]
        >= MIN_CONTROL_COMPUTED_EGFR_MEASUREMENTS
    ).all()

    print(
        f"{landmark_name}: QC passed "
        f"(N={len(cohort):,}, "
        f"cases={cases.sum():,}, "
        f"controls={controls.sum():,})"
    )


for name, cohort in landmark_cohorts.items():
    validate_landmark_cohort(
        cohort,
        name,
    )


In [ ]:
# Compact audit table for the four risk sets.
summary_rows = []

for name, cohort in landmark_cohorts.items():
    cases = cohort["dkd_binary"].eq(1)
    controls = cohort["dkd_binary"].eq(0)

    summary_rows.append(
        {
            "landmark": name,
            "landmark_days": int(
                cohort["landmark_days"].iloc[0]
            ),
            "n_total": len(cohort),
            "n_cases": int(cases.sum()),
            "n_controls": int(controls.sum()),
            "case_rate": float(
                cohort["dkd_binary"].mean()
            ),
            "median_case_time_from_t2d_days": float(
                cohort.loc[
                    cases,
                    "days_kidney_after_t2d",
                ].median()
            ),
            "median_case_time_after_landmark_days": float(
                cohort.loc[
                    cases,
                    "days_landmark_to_dkd",
                ].median()
            ),
            "median_control_followup_after_landmark_days": float(
                cohort.loc[
                    controls,
                    "followup_days_after_landmark",
                ].median()
            ),
            "min_control_followup_after_landmark_days": float(
                cohort.loc[
                    controls,
                    "followup_days_after_landmark",
                ].min()
            ),
        }
    )

landmark_summary = pd.DataFrame(
    summary_rows
)

landmark_summary


In [ ]:
for name, cohort in landmark_cohorts.items():
    save_gcs_parquet(
        cohort,
        f"{CURATED_DIR}/dkd_landmark_{name}.parquet",
    )

save_gcs_csv(
    landmark_summary,
    f"{CURATED_DIR}/landmark_cohort_summary.csv",
)


# 5. Feature Construction

No predictor may use information recorded on or after the prediction landmark.

Timing rules are applied according to feature type:

- Laboratory, vital, and body-measurement summaries use records from T2D diagnosis up to, but not including, the landmark.
- Comorbidity and medication history may include events before T2D, but only if they occur strictly before the landmark.
- Lifestyle, social-determinant, basic, and family-history survey responses are used only when the survey was completed strictly before the landmark.
- Polygenic risk scores and date of birth are treated as static attributes.

These rules keep the outcome and all time-varying predictors temporally separated.


## Prevalent Comorbidities

Comorbidities are defined from OMOP condition concepts and their descendants. A condition is considered prevalent only when its first observed event occurs strictly before the participant's landmark.

Broad CKD/DKD concepts are excluded from this predictor block to avoid encoding the outcome as a comorbidity feature.


In [ ]:
def _chunk_list(values, size=50_000):
    for i in range(0, len(values), size):
        yield values[i:i + size]


def get_descendant_concept_ids(ancestor_id):
    sql = f"""
    SELECT descendant_concept_id
    FROM `{CDR_DATASET}.concept_ancestor`
    WHERE ancestor_concept_id = @ancestor_id
    """

    config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "ancestor_id",
                "INT64",
                int(ancestor_id),
            )
        ]
    )

    return (
        bq_client.query(
            sql,
            job_config=config,
        )
        .to_dataframe()["descendant_concept_id"]
        .tolist()
    )


def get_condition_events_for_ancestor(
    cohort_ids,
    descendant_ids,
):
    """Pull dated condition events for the specified cohort."""
    sql = f"""
    SELECT
        person_id,
        condition_start_datetime
    FROM `{CDR_DATASET}.condition_occurrence`
    WHERE person_id IN UNNEST(@cohort_ids)
      AND condition_concept_id IN UNNEST(@descendant_ids)
    """

    frames = []

    for chunk in _chunk_list(cohort_ids):
        config = bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ArrayQueryParameter(
                    "cohort_ids",
                    "INT64",
                    chunk,
                ),
                bigquery.ArrayQueryParameter(
                    "descendant_ids",
                    "INT64",
                    descendant_ids,
                ),
            ]
        )

        x = bq_client.query(
            sql,
            job_config=config,
        ).to_dataframe()

        x["condition_start_datetime"] = parse_datetime(
            x["condition_start_datetime"]
        )

        frames.append(x)

    if not frames:
        return pd.DataFrame(
            columns=[
                "person_id",
                "condition_start_datetime",
            ]
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )


def generate_prevalent_comorbidities(
    cohort_df,
    ancestor_ids,
):
    """
    Generate one prevalent-before-landmark flag for each condition group.
    """
    cohort_ids = (
        cohort_df["person_id"]
        .astype(int)
        .tolist()
    )

    backbone = cohort_df[
        ["person_id", "cutoff_date"]
    ].copy()

    backbone["cutoff_date"] = parse_datetime(
        backbone["cutoff_date"]
    )

    flags = backbone[["person_id"]].copy()
    dates = backbone[["person_id"]].copy()
    mapping = []

    for label, ancestors in ancestor_ids.items():
        if not isinstance(ancestors, list):
            ancestors = [ancestors]

        feature_name = label.lower()
        event_frames = []

        for ancestor_id in ancestors:
            descendants = get_descendant_concept_ids(
                ancestor_id
            )

            if not descendants:
                continue

            events = get_condition_events_for_ancestor(
                cohort_ids,
                descendants,
            )

            if not events.empty:
                event_frames.append(events)

            mapping.append(
                {
                    "condition_label": label,
                    "ancestor_concept_id": int(
                        ancestor_id
                    ),
                    "column_name": feature_name,
                }
            )

        if not event_frames:
            flags[feature_name] = 0
            dates[
                f"{feature_name}_first_date"
            ] = pd.NaT
            continue

        events = pd.concat(
            event_frames,
            ignore_index=True,
        )

        events = events.merge(
            backbone,
            on="person_id",
            how="inner",
        )

        # Predictor events must be strictly before prediction.
        events = events.loc[
            events["condition_start_datetime"]
            < events["cutoff_date"]
        ]

        first_event = (
            events
            .groupby("person_id", as_index=False)[
                "condition_start_datetime"
            ]
            .min()
            .rename(
                columns={
                    "condition_start_datetime":
                    f"{feature_name}_first_date"
                }
            )
        )

        first_event[feature_name] = 1

        flags = flags.merge(
            first_event[
                ["person_id", feature_name]
            ],
            on="person_id",
            how="left",
        )

        dates = dates.merge(
            first_event[
                [
                    "person_id",
                    f"{feature_name}_first_date",
                ]
            ],
            on="person_id",
            how="left",
        )

        flags[feature_name] = (
            flags[feature_name]
            .fillna(0)
            .astype("int8")
        )

        del events, event_frames
        gc.collect()

    return flags, dates, pd.DataFrame(mapping)


In [ ]:
# Full comorbidity ancestor list for DKD risk prediction in a T2D cohort
# - Includes cardiometabolic, vascular, renal/urologic (non-CKD), liver, autoimmune,
#   diabetes complications, infectious, and mental/behavioral health.
# - Intentionally EXCLUDES broad CKD / kidney disease ancestors to avoid leakage.

ancestor_ids = {
    # -----------------------
    # 🩺 Cardiovascular / Vascular
    # -----------------------
    "Hypertension": [320128],                     # Hypertensive disorder / essential HTN
    "Coronary_Artery_Disease": [317576],          # Coronary arteriosclerosis
    "Myocardial_Infarction": [312327],            # Acute myocardial infarction (optional but reasonable)
    "Heart_Failure": [316139],                    # Heart failure
    "Atrial_Fibrillation": [313217],              # Atrial fibrillation
    "Peripheral_Arterial_Disease": [3654996],     # Peripheral arterial disease

    # -----------------------
    # 💊 Metabolic / Lifestyle-linked
    # -----------------------
    "Obesity": [433736],                          # Obesity
    "Gout": [440674],                              # Articular gout
    "Metabolic_Syndrome": [436940],             

    # -----------------------
    # 🧫 Renal / Urologic (non-CKD; avoids outcome leakage)
    # -----------------------
    "Acute_Kidney_Injury": [197320],              # Acute renal failure syndrome / AKI
    "Nephrolithiasis": [201620],                  # Kidney stone
    "UTI": [81902],                               # Urinary tract infectious disease

    # -----------------------
    # 🌬️ Pulmonary / Sleep
    # -----------------------
    "Sleep_Apnea": [313459],                      # Sleep apnea (metabolic + renal relevance)
    "COPD": [255573],                             # Chronic obstructive lung disease (optional but common enough)
    
    # -----------------------
    # 🍷 Liver
    # -----------------------
    "Steatotic_Liver_Disease": [4059290],       # Metabolic dysfunction-associated steatotic liver disease
    "Chronic_Liver_Disease": [4212540],         # Chronic liver disease (optional if MASLD behaves well)
    "Cirrhosis": [4064161],                     # Optional: can be rare/noisy
    "Chronic_Hepatitis_C": [198964],            # Optional: confounding control

    # -----------------------
    # 🦠 Autoimmune / Inflammatory
    # -----------------------
    "Rheumatoid_Arthritis": [80809],              # Rheumatoid arthritis
    "SLE": [257628],                              # Systemic lupus erythematosus
    "Psoriasis": [140168],                      # Optional: inflammation proxy

    # -----------------------
    # 🩸 Diabetes complications (microvascular / foot)
    # -----------------------
    "Diabetic_Retinopathy": [4174977],            # Retinopathy due to diabetes mellitus
    "Diabetic_Neuropathy": [37016354],            # Neuropathy due to type 2 diabetes mellitus
    "Diabetic_Foot": [4159742],                   # Diabetic foot ulcer

    # -----------------------
    # 🦠 Infectious (important confounders)
    # -----------------------
    "HIV": [439727],                              # Human immunodeficiency virus infection

    # -----------------------
    # 🧠 Mental / Behavioral Health
    # -----------------------
    "Major_Depressive_Disorder": [4152280],                  # Major depressive disorder (broad mood/depression signal)
    "Anxiety_Disorder": [442077],                # Anxiety disorder (broad)
    "Tobacco_Dependence_Syndrome": [437264],             # Tobacco dependence syndrome
    "Alcohol_Dependence": [435243],            # Alcohol dependence
    "Psychotic_Disorder": [436073],           # Optional: schizophrenia spectrum / psychotic disorders
}


In [ ]:
comorbidity_blocks = {}

for name, cohort in landmark_cohorts.items():
    print(f"\nBuilding comorbidities: {name}")

    flags, dates, mapping = (
        generate_prevalent_comorbidities(
            cohort,
            ancestor_ids,
        )
    )

    ensure_person_level(
        flags,
        f"comorbidities_{name}",
    )

    comorbidity_blocks[name] = flags

    save_gcs_parquet(
        flags,
        f"{CURATED_DIR}/comorbidities_{name}.parquet",
    )

# The concept mapping is identical across landmarks.
save_gcs_csv(
    mapping,
    f"{CURATED_DIR}/comorbidity_concept_mapping.csv",
)


## Laboratory Features

Laboratory features summarize measurements between T2D diagnosis and the landmark. Measurements on or after the landmark are excluded.

Snapshot and longitudinal summaries are generated from the same per-person timing rule.


In [ ]:
def get_lab_by_ancestor_all(ancestor_id):
    
    client = bq_client
    
    """
    Query labs for all participants in AoU by ancestor concept_id.
    Returns measurement rows with standard concept info and unit names.
    
    Parameters
    ----------
    ancestor_id : int
        OMOP concept_id for lab ancestor (e.g., HbA1c = 3004410)
    date_cutoffs : pd.DataFrame or None
        Must include person_id and cutoff_date (DKD or censoring).
        If provided, restricts to measurements before cutoff.
    """
    # --- Get descendants ---
    sql_desc = f"""
    SELECT descendant_concept_id
    FROM `{CDR_DATASET}.concept_ancestor`
    WHERE ancestor_concept_id = @anc_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("anc_id", "INT64", int(ancestor_id))]
    )
    desc = client.query(sql_desc, job_config=job_config).to_dataframe()["descendant_concept_id"].tolist()

    # --- Query measurement + concept names ---
    sql = f"""
    SELECT
      m.person_id,
      m.measurement_concept_id,
      m.measurement_datetime,
      m.value_as_number,
      m.unit_concept_id,
      c.concept_name   AS standard_concept_name,
      c.vocabulary_id  AS standard_vocabulary,
      u.concept_name   AS unit_concept_name
    FROM `{CDR_DATASET}.measurement` m
    JOIN `{CDR_DATASET}.concept` c
      ON m.measurement_concept_id = c.concept_id
    LEFT JOIN `{CDR_DATASET}.concept` u
      ON m.unit_concept_id = u.concept_id
    WHERE m.measurement_concept_id IN UNNEST(@desc_ids)
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("desc_ids", "INT64", desc)]
    )
    df = client.query(sql, job_config=job_config).to_dataframe()
    df["measurement_datetime"] = pd.to_datetime(df["measurement_datetime"], errors="coerce", utc=True)

    return df


In [ ]:
def clean_labs(df, lab_name, low=None, high=None, unit_map=None, drop_outliers=True):
    """
    Flexible lab QC function.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must include person_id, measurement_datetime, value_as_number, unit_concept_name
    lab_name : str
        Label for logging/plots (e.g., "Fasting Glucose")
    low, high : float or None
        Plausible range. Values outside are set to NaN (or dropped if drop_outliers=True).
    unit_map : dict or None
        Mapping of {unit_concept_name: factor}. Values multiplied by factor to harmonize.
    drop_outliers : bool
        If True, drop implausible values. If False, keep but set to NaN.
    
    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with harmonized units and flagged implausible values.
    """
    df = df.copy()
    
    # --- Unit harmonization ---
    if unit_map:
        ref_unit = next(iter(unit_map.keys()))  # first key is reference
        for u, factor in unit_map.items():
            mask = df["unit_concept_name"] == u
            if mask.any():
                df.loc[mask, "value_as_number"] = df.loc[mask, "value_as_number"] * factor
                df.loc[mask, "unit_concept_name"] = ref_unit

    # --- Range QC ---
    if low is not None and high is not None:
        outlier_mask = (df["value_as_number"] < low) | (df["value_as_number"] > high)
        if drop_outliers:
            df = df.loc[~outlier_mask].copy()
        else:
            df.loc[outlier_mask, "value_as_number"] = np.nan

    return df

def clean_and_filter_units(df, lab_name, drop_units=None,
                           low=None, high=None,
                           unit_map=None, drop_outliers=False):
    """
    Drop unwanted units and clean lab values.

    Parameters
    ----------
    df : pd.DataFrame
        Lab dataframe with columns [person_id, measurement_datetime, value_as_number, unit_concept_name].
    lab_name : str
        Short label for the lab (e.g., "LDL cholesterol").
    drop_units : list[str] or None
        Units to drop entirely before cleaning.
    low, high : float or None
        Plausible value range. Values outside set to NaN (or dropped if drop_outliers=True).
    unit_map : dict or None
        Mapping of {unit_concept_name: factor} for harmonization.
    drop_outliers : bool
        If True, drop implausible rows instead of setting to NaN.

    Returns
    -------
    pd.DataFrame
        Cleaned and filtered dataframe.
    """

    df = df.copy()

    # 1. Drop unwanted units
    if drop_units:
        df = df[~df["unit_concept_name"].isin(drop_units)].copy()

    # 2. Apply clean_labs function
    df = clean_labs(
        df,
        lab_name=lab_name,
        low=low,
        high=high,
        unit_map=unit_map,
        drop_outliers=drop_outliers
    )

    return df


In [ ]:
# ==========================================================
# 🧪 Flexible Lab Summarization (per-lab configurable)
# ==========================================================
# This is a drop-in replacement for summarize_single_lab() that lets you decide,
# per lab, exactly which features to compute (last/mean/sd/cov/slope/min/max/n/recency/etc)
# plus clinically interpretable abnormality features.
#
# Key ideas:
# - Always respects cutoff_date to prevent leakage (case/control-aware censoring).
# - Uses a per-lab "spec" dict to define which summaries to compute.
# - Produces stable, model-friendly columns with consistent naming.
# ==========================================================

from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

def compute_slope(group, date_col, value_col):
    """Slope of lab values over time (years) using simple linear regression."""
    g = group[[date_col, value_col]].dropna()
    if len(g) <= 1:
        return np.nan

    # Time since first measurement in years
    t = (g[date_col] - g[date_col].min()).dt.days.values.reshape(-1, 1) / 365.25
    vals = g[value_col].values

    mask = (~np.isnan(t.flatten())) & (~np.isnan(vals))
    if mask.sum() <= 1:
        return np.nan

    model = LinearRegression().fit(t[mask].reshape(-1, 1), vals[mask])
    return float(model.coef_[0])


def _apply_cutoff_filter(df, cutoffs, id_col, date_col):
    """
    Merge cutoff_date and filter measurements strictly before cutoff_date.
    Keeps only people present in cutoffs (inner join) to avoid accidental leakage.
    """
    if cutoffs is None:
        return df

    base = cutoffs[[id_col, "cutoff_date"]].copy()
    base["cutoff_date"] = pd.to_datetime(base["cutoff_date"], errors="coerce", utc=True)

    out = df.merge(base, on=id_col, how="inner")
    out = out[out[date_col] < out["cutoff_date"]]
    return out


def _ensure_datetime(df, date_col):
    """Ensure date_col is a timezone-aware datetime."""
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    return df


# ==========================================================
# 🧪 Flexible Lab Summarization (per-lab configurable)
# ==========================================================
# What this does
# -------------
# Given a longitudinal lab/measurement table (multiple rows per person over time),
# this function creates a single person-level feature table using a spec dict.
#
# Core principles
# ---------------
# 1) ✅ Anti-leakage censoring:
#    If you pass `cutoffs` (person_id + cutoff_date), we only use measurements
#    strictly BEFORE cutoff_date for each person.
#
# 2) ✅ Snapshot + longitudinal features:
#    - Snapshot: last value before cutoff, recency to last measurement
#    - Longitudinal: n, mean, sd, cov, min/max, span, slope, fraction abnormal
#
# 3) ✅ Correct handling of missing labs:
#    Output includes ALL persons from `cutoffs`, even if the lab is missing.
#    That means *_n can correctly be 0, and missingness is explicit ensure.
#
# 4) ✅ Stable, model-friendly columns:
#    Feature names are consistently prefixed with `lab_label` (e.g., egfr_last).
#
# Notes
# -----
# - XGBoost can handle NaNs for continuous features.
# - We fill missing *_n with 0, and fill binary flags (ever_abnormal, slope_dir) with 0.
# ==========================================================


from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np


def compute_slope(group: pd.DataFrame, date_col: str, value_col: str) -> float:
    """
    Fit a simple linear regression of value ~ time (years since first measurement).
    Returns slope (units per year).
    """
    g = group[[date_col, value_col]].dropna()
    if len(g) <= 1:
        return np.nan

    # time in years from first measurement
    t = (g[date_col] - g[date_col].min()).dt.days.values.reshape(-1, 1) / 365.25
    vals = g[value_col].values

    mask = (~np.isnan(t.flatten())) & (~np.isnan(vals))
    if mask.sum() <= 1:
        return np.nan

    model = LinearRegression().fit(t[mask].reshape(-1, 1), vals[mask])
    return float(model.coef_[0])


def _ensure_datetime_utc(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    Ensure date_col is timezone-aware UTC datetime.
    Safe even if input is date-only or already tz-aware.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    return df


def _apply_cutoff_filter(
    df: pd.DataFrame,
    cutoffs: pd.DataFrame | None,
    id_col: str,
    date_col: str,
    cutoff_col: str = "cutoff_date",
    inclusive: bool = False,   # if True, keep measurements <= cutoff_date
) -> pd.DataFrame:
    """
    Merge cutoff_date and filter measurements relative to cutoff_date.

    - Uses INNER join to ensure we only keep people present in cutoffs.
    - Filters strictly before cutoff by default (anti-leakage).
    """
    if cutoffs is None:
        return df

    base = cutoffs[[id_col, cutoff_col]].copy()
    base[cutoff_col] = pd.to_datetime(base[cutoff_col], errors="coerce", utc=True)

    out = df.merge(base, on=id_col, how="inner")

    if inclusive:
        out = out[out[date_col] <= out[cutoff_col]]
    else:
        out = out[out[date_col] < out[cutoff_col]]

    return out


def summarize_lab_flexible(
    lab_df: pd.DataFrame,
    lab_label: str,
    spec: dict,
    cutoffs: pd.DataFrame | None = None,
    id_col: str = "person_id",
    date_col: str | None = None,
    value_col: str = "value_as_number",
    cutoff_col: str = "cutoff_date",
    inclusive_cutoff: bool = False,
) -> pd.DataFrame:
    """
    Flexible lab summarization controlled by a per-lab spec dict.

    Parameters
    ----------
    lab_df
        Long-format lab table with columns: id_col, date_col (or auto-detect),
        and value_col (numeric).
    lab_label
        Prefix used for feature names (e.g. "egfr", "uacr", "whr").
    spec
        Dict controlling which features to compute. Example:
            spec = {
                "compute": ["last", "mean", "sd", "cov", "min", "max", "n",
                            "recency", "slope", "slope_dir", "span",
                            "ever_abnormal", "frac_abnormal"],
                "abnormal": {"direction": "low", "low": 60},
                "min_n_for_slope": 3,
                "slope_eps": 0.0,
            }
    cutoffs
        Person-level censoring table containing [id_col, cutoff_col].
        If provided, output will include ALL people in this table.
    date_col
        If None, tries "measurement_datetime" then "measurement_date".
    value_col
        Name of numeric value column.
    cutoff_col
        Column name in `cutoffs` for censoring date.
    inclusive_cutoff
        If True, keep measurements on the cutoff day (<=). Default False uses strict <.

    Returns
    -------
    Person-level pd.DataFrame:
        Always includes id_col.
        If cutoffs is provided, includes all people in cutoffs even if lab is missing.
        *_n is filled to 0 when missing.
        Binary columns are filled to 0 for stability.
    """

    # ---------------------------
    # 0) Copy and detect date column
    # ---------------------------
    df = lab_df.copy()

    if date_col is None:
        if "measurement_datetime" in df.columns:
            date_col = "measurement_datetime"
        elif "measurement_date" in df.columns:
            date_col = "measurement_date"
        else:
            raise ValueError(
                "No date column found. Expected 'measurement_datetime' or 'measurement_date', "
                "or pass date_col explicitly."
            )

    # ---------------------------
    # 1) Build the output base FIRST
    #    This is the key fix: include everyone even if lab is missing
    # ---------------------------
    if cutoffs is not None:
        out = cutoffs[[id_col]].drop_duplicates().copy()
        out[id_col] = pd.to_numeric(out[id_col], errors="coerce")
        out = out.dropna(subset=[id_col])
        out[id_col] = out[id_col].astype("int64")
        out = out.sort_values(id_col).reset_index(drop=True)
    else:
        # If no cutoffs provided, we can only output people present in the lab table
        out = pd.DataFrame({id_col: pd.to_numeric(df[id_col], errors="coerce")})
        out = out.dropna(subset=[id_col])
        out[id_col] = out[id_col].astype("int64")
        out = pd.DataFrame({id_col: out[id_col].unique()}).sort_values(id_col).reset_index(drop=True)

    # ---------------------------
    # 2) Normalize datetime to UTC and apply censoring
    # ---------------------------
    df = _ensure_datetime_utc(df, date_col)

    if cutoffs is not None:
        # ensure cutoffs has UTC datetime
        cutoffs_local = cutoffs[[id_col, cutoff_col]].copy()
        cutoffs_local[cutoff_col] = pd.to_datetime(cutoffs_local[cutoff_col], errors="coerce", utc=True)

        df = _apply_cutoff_filter(
            df=df,
            cutoffs=cutoffs_local,
            id_col=id_col,
            date_col=date_col,
            cutoff_col=cutoff_col,
            inclusive=inclusive_cutoff,
        )

    # ---------------------------
    # 3) If nothing remains, return the base (everyone present, all features missing)
    # ---------------------------
    if df.empty:
        # still stable: if user asked for n, provide it as 0
        compute = set(spec.get("compute", ["last"]))
        if "n" in compute:
            out[f"{lab_label}_n"] = 0
            out[f"{lab_label}_n"] = out[f"{lab_label}_n"].astype("Int64")
        return out

    # ---------------------------
    # 4) Clean and sort the working df
    # ---------------------------
    df = df[[id_col, date_col, value_col]].copy()
    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[id_col, date_col, value_col])

    df[id_col] = df[id_col].astype("int64")
    df = df.sort_values([id_col, date_col], kind="mergesort")

    compute = set(spec.get("compute", ["last"]))  # default last only
    min_n_for_slope = int(spec.get("min_n_for_slope", 3))
    slope_eps = float(spec.get("slope_eps", 0.0))

    g = df.groupby(id_col)[value_col]

    # ---------------------------
    # 5) Core numeric summaries
    # ---------------------------
    if "n" in compute:
        n_df = g.size().rename(f"{lab_label}_n").reset_index()
        out = out.merge(n_df, on=id_col, how="left")
        out[f"{lab_label}_n"] = out[f"{lab_label}_n"].fillna(0).astype("Int64")

    if "last" in compute:
        last_df = df.groupby(id_col)[value_col].last().rename(f"{lab_label}_last").reset_index()
        out = out.merge(last_df, on=id_col, how="left")

    if "first" in compute:
        first_df = df.groupby(id_col)[value_col].first().rename(f"{lab_label}_first").reset_index()
        out = out.merge(first_df, on=id_col, how="left")

    if "mean" in compute:
        out = out.merge(g.mean().rename(f"{lab_label}_mean").reset_index(), on=id_col, how="left")

    if "sd" in compute:
        out = out.merge(g.std().rename(f"{lab_label}_sd").reset_index(), on=id_col, how="left")

    if "min" in compute:
        out = out.merge(g.min().rename(f"{lab_label}_min").reset_index(), on=id_col, how="left")

    if "max" in compute:
        out = out.merge(g.max().rename(f"{lab_label}_max").reset_index(), on=id_col, how="left")

    if "median" in compute:
        out = out.merge(g.median().rename(f"{lab_label}_median").reset_index(), on=id_col, how="left")

    if "cov" in compute:
        # ensure mean + sd exist
        mean_col = f"{lab_label}_mean"
        sd_col = f"{lab_label}_sd"
        if mean_col not in out.columns:
            out = out.merge(g.mean().rename(mean_col).reset_index(), on=id_col, how="left")
        if sd_col not in out.columns:
            out = out.merge(g.std().rename(sd_col).reset_index(), on=id_col, how="left")
        out[f"{lab_label}_cov"] = out[sd_col] / out[mean_col].replace(0, np.nan)

    # ---------------------------
    # 6) Measurement timing features
    # ---------------------------
    if "span" in compute:
        # span_days = last_date - first_date
        span_df = df.groupby(id_col)[date_col].agg(["min", "max"]).reset_index()
        span_df[f"{lab_label}_span_days"] = (span_df["max"] - span_df["min"]).dt.days
        out = out.merge(span_df[[id_col, f"{lab_label}_span_days"]], on=id_col, how="left")

    if "recency" in compute:
        # recency_days = cutoff_date - last_measurement_date
        last_dt = df.groupby(id_col)[date_col].max().rename("last_dt").reset_index()

        if cutoffs is not None:
            base = cutoffs[[id_col, cutoff_col]].copy()
            base[cutoff_col] = pd.to_datetime(base[cutoff_col], errors="coerce", utc=True)

            rec = last_dt.merge(base, on=id_col, how="left")
            rec[f"{lab_label}_recency_days"] = (rec[cutoff_col] - rec["last_dt"]).dt.days
        else:
            # no cutoffs: define recency relative to each person's own last measurement
            rec = last_dt.copy()
            rec[f"{lab_label}_recency_days"] = 0

        out = out.merge(rec[[id_col, f"{lab_label}_recency_days"]], on=id_col, how="left")

    # ---------------------------
    # 7) Trajectory: slope + optional slope direction
    # ---------------------------
    if "slope" in compute or "slope_dir" in compute:
        counts = df.groupby(id_col).size()
        eligible_ids = counts[counts >= min_n_for_slope].index

        slopes = []
        for pid, grp in df[df[id_col].isin(eligible_ids)].groupby(id_col):
            slopes.append((pid, compute_slope(grp, date_col, value_col)))

        slope_df = pd.DataFrame(slopes, columns=[id_col, f"{lab_label}_slope"])
        out = out.merge(slope_df, on=id_col, how="left")

        if "slope_dir" in compute:
            s = out[f"{lab_label}_slope"]
            out[f"{lab_label}_slope_dir"] = np.where(
                s.isna(), 0,
                np.where(s > slope_eps, 1, np.where(s < -slope_eps, -1, 0))
            ).astype(int)

    # ---------------------------
    # 8) Abnormality features (ever abnormal + fraction abnormal)
    # ---------------------------
    abn = spec.get("abnormal", None)
    if abn and (("ever_abnormal" in compute) or ("frac_abnormal" in compute)):
        direction = abn.get("direction", "both")  # "high", "low", "both"
        low_thr = abn.get("low", None)
        high_thr = abn.get("high", None)

        # row-level abnormal mask
        ab_mask = pd.Series(False, index=df.index)

        if direction in ["low", "both"] and low_thr is not None:
            ab_mask = ab_mask | (df[value_col] < float(low_thr))
        if direction in ["high", "both"] and high_thr is not None:
            ab_mask = ab_mask | (df[value_col] > float(high_thr))

        df_ab = df.assign(_ab=ab_mask.astype(int))

        if "ever_abnormal" in compute:
            ever = df_ab.groupby(id_col)["_ab"].max().rename(f"{lab_label}_ever_abnormal").reset_index()
            out = out.merge(ever, on=id_col, how="left")

        if "frac_abnormal" in compute:
            frac = df_ab.groupby(id_col)["_ab"].mean().rename(f"{lab_label}_frac_abnormal").reset_index()
            out = out.merge(frac, on=id_col, how="left")

    # ---------------------------
    # 9) Final cleanup
    # ---------------------------
    # - leave continuous NaNs as-is
    # - *_n is already filled to 0 above
    # - slope_dir remains 0 when slope is missing (by design)
    # - abnormality features should be NA when n == 0

    n_col = f"{lab_label}_n"
    ever_col = f"{lab_label}_ever_abnormal"
    frac_col = f"{lab_label}_frac_abnormal"
    slope_dir_col = f"{lab_label}_slope_dir"

    # Ensure n exists if abnormality was requested (otherwise we cannot apply the rule)
    if (ever_col in out.columns or frac_col in out.columns) and (n_col not in out.columns):
        # compute n on the fly from df (post-censoring)
        n_df = df.groupby(id_col).size().rename(n_col).reset_index()
        out = out.merge(n_df, on=id_col, how="left")
        out[n_col] = out[n_col].fillna(0).astype("Int64")

    # Apply missingness semantics:
    # if no measurements, abnormality is undefined
    if n_col in out.columns:
        if ever_col in out.columns:
            out.loc[out[n_col] == 0, ever_col] = np.nan
            # keep 0/1 only when measured
            out.loc[out[n_col] > 0, ever_col] = out.loc[out[n_col] > 0, ever_col].fillna(0)
            out[ever_col] = out[ever_col].astype("Float64")  # allows NA + 0/1

        if frac_col in out.columns:
            out.loc[out[n_col] == 0, frac_col] = np.nan  # leave fractions as float with NA

    # Keep slope_dir stable if you want (NA slope gives 0 direction)
    if slope_dir_col in out.columns:
        out[slope_dir_col] = out[slope_dir_col].fillna(0).astype(int)

    return out


In [ ]:
# ==========================================================
# ✅ LAB_SPECS: DKD-ready, clinically interpretable, per-lab configs
# ==========================================================
# Notes
# - FULL trajectory (slope + variability): A1c, creatinine, eGFR, urine albumin/UACR, BP
# - LIMITED trajectory (no slope): electrolytes, BUN, hemoglobin, lipids, uric acid, HR
# - OGTT is often sparse → keep minimal
# - Excludes: insulin, glucagon, c-peptide 
#
# Thresholds are "clinical defaults" and can be refined later based on your unit harmonization.
# ==========================================================

LAB_SPECS = {
    # -----------------------
    # 🩸 Glycemic
    # -----------------------
    "hba1c": {
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 7.0},
        "min_n_for_slope": 3,
        "slope_eps": 0.0,
    },
    "fbg": { # very sparse so keep the min features
        "compute": ["last", "mean", "min", "max", "n", "recency",
                    "ever_abnormal", "frac_abnormal"],
    },
    "rbg": {  # random blood glucose (mg/dL)
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 200},
        "min_n_for_slope": 3,
    },

    # -----------------------
    # 🧫 Renal
    # -----------------------
    "serum_creatinine": {  # mg/dL
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 1.3},  # generic adult flag; refine by sex if desired
        "min_n_for_slope": 3,
    },
    "egfr": {  # mL/min/1.73m2
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "low", "low": 60},
        "min_n_for_slope": 3,
    },
    "cystatin_c": {  # mg/L; can be sparse
        "compute": ["last", "mean", "min", "max", "n", "recency", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 1.0},  # rough; adjust after unit QC
        "min_n_for_slope": 3,
    },
    "urine_albumin": {  # typically mg/L or mg/dL depending; after harmonization
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 30},  # placeholder; refine per units (often mg/L)
        "min_n_for_slope": 3,
    },
    "uacr": {  # mg/g
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 30},
        "min_n_for_slope": 3,
    },

    # -----------------------
    # 🧂 Electrolytes / Acid–base
    # -----------------------
    "na": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 135, "high": 145},
    },
    "k": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 3.5, "high": 5.0},
    },
    "cl": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 98, "high": 106},
    },
    "hco3": {  # bicarbonate / CO2
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 22, "high": 29},
    },
    "ca": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 8.5, "high": 10.5},
    },
    "mg": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 1.6, "high": 2.6},
    },
    "po4": {  # phosphate
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 2.5, "high": 4.5},
    },

    # -----------------------
    # 🧪 Other renal-adjacent
    # -----------------------
    "bun": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 20},
    },

    # -----------------------
    # 🩸 Hematology
    # -----------------------
    "hemoglobin": {
        "compute": ["last", "mean", "min", "n", "recency", "ever_abnormal", "frac_abnormal"],
        # Generic anemia threshold; refine by sex if you want (male<13, female<12)
        "abnormal": {"direction": "low", "low": 12.0},
    },

    # -----------------------
    # 🧈 Lipids
    # -----------------------
    "ldl": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 130},
    },
    "hdl": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "low", "low": 40},
    },
    "tg": {  # triglycerides
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 150},
    },
    "total_chol": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 200},
    },

    # -----------------------
    # 🧫 Liver function
    # -----------------------
    "alt": {  # U/L
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 40},
    },
    "ast": {  # U/L
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 40},
    },
    "alp": {  # U/L
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 120},
    },
    "total_bilirubin": {  # mg/dL
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 1.2},
    },
    "direct_bilirubin": {  # mg/dL
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 0.3},
    },
    "albumin": {  # g/dL
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "low", "low": 3.5},
    },
    "total_protein": {  # g/dL
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "both", "low": 6.0, "high": 8.3},
    },
    

    # -----------------------
    # 🧂 Uric acid
    # -----------------------
    "uric_acid": {
        "compute": ["last", "mean", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 7.0},
    },

    
    # -----------------------
    # 🔥 Inflammation
    # -----------------------
    "esr": {  # mm/hr
        "compute": [
            "last",
            "mean",
            "min",
            "max",
            "n",
            "recency",
            "ever_abnormal",
            "frac_abnormal",
        ],
        # Conservative adult threshold
        "abnormal": {"direction": "high", "high": 20},
    },

    
    # -----------------------
    # 🩺 Vitals (treat like labs; derived from measurement table)
    # -----------------------
    "sbp": {
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 140},
        "min_n_for_slope": 3,
    },
    "dbp": {
        "compute": ["last", "mean", "sd", "cov", "min", "max", "n", "recency", "slope", "slope_dir",
                    "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 90},
        "min_n_for_slope": 3,
    },
    "hr": {
        "compute": ["last", "mean", "sd", "min", "max", "n", "recency", "ever_abnormal", "frac_abnormal"],
        "abnormal": {"direction": "high", "high": 100},
    },
}


In [ ]:
import pandas as pd
import numpy as np

def build_single_lab_features_at_cutoff(
    cutoff_df: pd.DataFrame,
    lab_df: pd.DataFrame,
    lab_key: str,
    lab_specs: dict,
    lab_label_prefix: str,
    id_col: str = "person_id",
    lab_date_col: str = "measurement_datetime",
    lab_value_col: str = "value_as_number",
    t2d_col: str = "t2d_first_evidence_date",
    cutoff_col: str = "cutoff_date",
    require_after_t2d: bool = True,
    strict_before_cutoff: bool = True,
):
    """
    Builds person-level features for ONE lab at a landmark cutoff.

    Key fix:
      - passes per-person cutoff_date into summarize_lab_flexible via cutoffs=...
        so *_recency_days is computed correctly.
    """

    if lab_key not in lab_specs:
        raise KeyError(f"lab_key='{lab_key}' not found in lab_specs")

    # ---- prep cutoff frame (anchor)
    base = cutoff_df[[id_col, t2d_col, cutoff_col]].copy()
    base[id_col] = pd.to_numeric(base[id_col], errors="coerce").astype("Int64")
    base[t2d_col] = pd.to_datetime(base[t2d_col], errors="coerce", utc=True)
    base[cutoff_col] = pd.to_datetime(base[cutoff_col], errors="coerce", utc=True)
    base = base.dropna(subset=[id_col, t2d_col, cutoff_col])

    # summarize_lab_flexible expects this exact name
    cutoffs = base[[id_col, cutoff_col]].rename(columns={cutoff_col: "cutoff_date"}).drop_duplicates()

    # ---- prep lab frame
    if id_col not in lab_df.columns:
        raise KeyError(f"lab_df missing id_col='{id_col}'")
    if lab_date_col not in lab_df.columns:
        raise KeyError(f"lab_df missing lab_date_col='{lab_date_col}'")
    if lab_value_col not in lab_df.columns:
        raise KeyError(f"lab_df missing lab_value_col='{lab_value_col}'")

    df = lab_df[[id_col, lab_date_col, lab_value_col]].copy()
    df = df.rename(columns={lab_date_col: "_lab_dt", lab_value_col: "_lab_val"})

    df[id_col] = pd.to_numeric(df[id_col], errors="coerce").astype("Int64")
    df["_lab_dt"] = pd.to_datetime(df["_lab_dt"], errors="coerce", utc=True)
    df["_lab_val"] = pd.to_numeric(df["_lab_val"], errors="coerce")
    df = df.dropna(subset=[id_col, "_lab_dt", "_lab_val"])

    # ---- restrict to cohort + window
    x = df.merge(base, on=id_col, how="inner")

    if require_after_t2d:
        x = x[x["_lab_dt"] >= x[t2d_col]]

    if strict_before_cutoff:
        x = x[x["_lab_dt"] < x[cutoff_col]]
    else:
        x = x[x["_lab_dt"] <= x[cutoff_col]]

    x = x[[id_col, "_lab_dt", "_lab_val"]].rename(
        columns={"_lab_dt": "measurement_datetime", "_lab_val": "value_as_number"}
    )

    # ---- summarize with cutoff anchor included (FIX)
    feats = summarize_lab_flexible(
        lab_df=x,
        lab_label=lab_label_prefix.rstrip("_"),
        spec=lab_specs[lab_key],
        cutoffs=cutoffs,
        id_col=id_col,
        date_col="measurement_datetime",
        value_col="value_as_number",
    )

    # ---- ensure full cohort retained
    feats = cutoffs[[id_col]].merge(feats, on=id_col, how="left")

    return feats


### Curated Glycemic and Renal Measurements

HbA1c, fasting glucose, random glucose, UACR, eGFR, serum creatinine, and spot urine albumin are loaded from notebook 01 rather than re-extracted.


In [ ]:
df_a1c = read_gcs_csv(INPUT_PATHS["a1c"])
df_fbg = read_gcs_csv(INPUT_PATHS["fbg"])
df_rbg = read_gcs_parquet(INPUT_PATHS["rbg"])

# Standardize IDs once.
for name, frame in {
    "HbA1c": df_a1c,
    "FBG": df_fbg,
    "RBG": df_rbg,
    "UACR": df_uacr_long,
    "eGFR": df_egfr_long,
    "SCr": df_scr_long,
    "UrAlb": df_ualb_long,
}.items():
    frame["person_id"] = pd.to_numeric(
        frame["person_id"],
        errors="coerce",
    ).astype("Int64")


In [ ]:
CORE_LAB_INPUTS = {
    "HbA1c": {
        "df": df_a1c,
        "lab_key": "hba1c",
        "value_col": "value_as_number",
    },
    "FBG": {
        "df": df_fbg,
        "lab_key": "fbg",
        "value_col": "value_as_number",
    },
    "RBG": {
        "df": df_rbg,
        "lab_key": "rbg",
        "value_col": "value_as_number",
    },
    "UACR": {
        "df": df_uacr_long,
        "lab_key": "uacr",
        "value_col": "uacr_mg_g",
    },
    "eGFR": {
        "df": df_egfr_long,
        "lab_key": "egfr",
        "value_col": "egfr_ml_min_1p73m2",
    },
    "Screatinine": {
        "df": df_scr_long,
        "lab_key": "serum_creatinine",
        "value_col": "scr_mg_dl",
    },
    "UrAlb": {
        "df": df_ualb_long,
        "lab_key": "urine_albumin",
        "value_col": "ualb_mg_dl",
    },
}


def build_core_lab_block(cutoff_df):
    """Build the curated glycemic and renal feature block."""
    backbone = cutoff_df[
        ["person_id"]
    ].drop_duplicates().copy()

    out = backbone

    for label, cfg in CORE_LAB_INPUTS.items():
        feats = build_single_lab_features_at_cutoff(
            cutoff_df=cutoff_df,
            lab_df=cfg["df"],
            lab_key=cfg["lab_key"],
            lab_specs=LAB_SPECS,
            lab_label_prefix=label,
            lab_date_col="measurement_datetime",
            lab_value_col=cfg["value_col"],
        )

        out = out.merge(
            feats,
            on="person_id",
            how="left",
            validate="one_to_one",
        )

    return out


core_lab_blocks = {}

for name, cohort in landmark_cohorts.items():
    print(f"\nBuilding curated core labs: {name}")

    block = build_core_lab_block(cohort)
    ensure_person_level(block, f"core_labs_{name}")

    core_lab_blocks[name] = block

    save_gcs_parquet(
        block,
        f"{CURATED_DIR}/core_labs_{name}.parquet",
    )


### Additional Laboratory, Vital, and Body Measurements

The remaining laboratory and measurement domains use the same OMOP ancestor concepts and clinical cleaning rules as the original extraction workflow. Each source is queried once, cleaned once, and summarized independently at each landmark.


In [ ]:
# Additional measurement definitions carried forward from the original workflow.
ADDITIONAL_MEASUREMENTS = {
    # Electrolytes
    "Na": {
        "ancestor": 3019550,
        "lab_key": "na",
        "low": 120,
        "high": 160,
        "drop_units": [
            "international unit per milliliter",
            "nanogram per deciliter",
        ],
    },
    "K": {
        "ancestor": 3023103,
        "lab_key": "k",
        "low": 2.5,
        "high": 7,
        "drop_units": ["microgram per liter"],
    },
    "Cl": {
        "ancestor": 3014576,
        "lab_key": "cl",
        "low": 80,
        "high": 120,
        "drop_units": [""],
    },
    "HCO3": {
        "ancestor": 3016293,
        "lab_key": "hco3",
        "low": 10,
        "high": 40,
        "drop_units": [""],
    },
    "Ca": {
        "ancestor": 3015377,
        "lab_key": "ca",
        "low": 6,
        "high": 14,
        "drop_units": ["millimole per liter"],
    },
    "Mg": {
        "ancestor": 3012095,
        "lab_key": "mg",
        "low": 1,
        "high": 4,
        "unit_map": {
            "milligram per deciliter": 1.0,
            "No matching concept": 1.0,
            "mEq/L": 1.0,
            "milliequivalent per liter": 1.0,
            "millimole per liter": 2.43,
        },
    },
    "PO4": {
        "ancestor": 3003458,
        "lab_key": "po4",
    },

    # Renal-adjacent / hematology
    "BUN": {
        "ancestor": 3013682,
        "lab_key": "bun",
        "low": 1,
        "high": 200,
        "unit_map": {
            "milligram per deciliter": 1.0,
            "millimole per liter": 2.801,
        },
    },
    "Hgb": {
        "ancestor": 3000963,
        "lab_key": "hemoglobin",
        "low": 2,
        "high": 25,
        "unit_map": {
            "gram per liter": 0.1,
        },
    },

    # Lipids
    "LDL": {
        "ancestor": 3028437,
        "lab_key": "ldl",
        "low": 10,
        "high": 600,
        "drop_units": [
            "nanomole per liter",
            "nanometer",
            "unit per liter",
        ],
    },
    "HDL": {
        "ancestor": 3007070,
        "lab_key": "hdl",
        "low": 5,
        "high": 150,
        "drop_units": [
            "calculated",
            "percent",
            "ratio",
            "unit",
        ],
    },
    "TG": {
        "ancestor": 3022192,
        "lab_key": "tg",
        "low": 10,
        "high": 2000,
        "drop_units": ["millimole per liter"],
    },
    "TotalChol": {
        "ancestor": 3027114,
        "lab_key": "total_chol",
        "low": 50,
        "high": 1000,
        "drop_units": ["calculated", "titer"],
    },

    # Liver
    "ALT": {
        "ancestor": 3006923,
        "lab_key": "alt",
        "low": 0,
        "high": 500,
        "drop_units": ["percent"],
        "unit_map": {
            "U/L": 1.0,
            "unit per liter": 1.0,
        },
    },
    "AST": {
        "ancestor": 3013721,
        "lab_key": "ast",
        "low": 0,
        "high": 500,
        "unit_map": {
            "U/L": 1.0,
            "unit per liter": 1.0,
        },
    },
    "ALP": {
        "ancestor": 3035995,
        "lab_key": "alp",
        "low": 20,
        "high": 1000,
        "drop_units": [
            "present",
            "microgram per liter",
        ],
        "unit_map": {
            "U/L": 1.0,
            "unit per liter": 1.0,
        },
    },
    "TotalBilirubin": {
        "ancestor": 3024128,
        "lab_key": "total_bilirubin",
        "low": 0,
        "high": 30,
        "unit_map": {
            "milligram per deciliter": 1.0,
            "mg/dL": 1.0,
            "micromole per liter": 0.0585,
        },
    },
    "DirectBilirubin": {
        "ancestor": 3027597,
        "lab_key": "direct_bilirubin",
        "low": 0,
        "high": 10,
        "unit_map": {
            "milligram per deciliter": 1.0,
            "mg/dL": 1.0,
            "micromole per liter": 0.0585,
        },
    },
    "Albumin": {
        "ancestor": 3024561,
        "lab_key": "albumin",
        "low": 2,
        "high": 6,
        "drop_units": ["unit per liter"],
        "unit_map": {
            "gram per deciliter": 1.0,
            "g/dL": 1.0,
            "gram per liter": 0.1,
            "milligram per milliliter": 0.1,
        },
    },
    "TotalProtein": {
        "ancestor": 3020630,
        "lab_key": "total_protein",
        "low": 4,
        "high": 10,
        "drop_units": ["millimole per liter"],
        "unit_map": {
            "gram per deciliter": 1.0,
            "g/dL": 1.0,
            "gram per liter": 0.1,
        },
    },

    # Other
    "UricAcid": {
        "ancestor": 3037556,
        "lab_key": "uric_acid",
        "low": 1,
        "high": 20,
        "drop_units": ["percent"],
    },
    "ESR": {
        "ancestor": 3015183,
        "lab_key": "esr",
        "low": 0,
        "high": 120,
        "drop_units": [
            "millimole per hour",
            "mm",
            "no value",
            "No matching concept",
        ],
        "drop_outliers": True,
    },

    # Vitals
    "SBP": {
        "ancestor": 3004249,
        "lab_key": "sbp",
        "low": 60,
        "high": 260,
        "drop_outliers": True,
    },
    "DBP": {
        "ancestor": 3012888,
        "lab_key": "dbp",
        "low": 30,
        "high": 150,
        "drop_outliers": True,
    },
    "HR": {
        "ancestor": 1004025,
        "lab_key": "hr",
        "low": 25,
        "high": 220,
        "drop_units": [
            "millisecond",
            "siemens",
            "percent",
        ],
        "drop_outliers": True,
    },
}

BODY_SPECS = {
    "weight": {
        "compute": [
            "last",
            "mean",
            "min",
            "max",
            "n",
            "recency",
            "slope",
            "slope_dir",
        ],
        "min_n_for_slope": 3,
        "slope_eps": 0.0,
    },
    "height": {
        "compute": ["last"],
    },
}

BODY_MEASUREMENTS = {
    "Weight": {
        "ancestor": 3025315,
        "lab_key": "weight",
        "low": 25,
        "high": 400,
        "drop_units": [
            "centimeter",
            "inch (international)",
            "ounce (avoirdupois)",
            "No matching concept",
        ],
        "unit_map": {
            "pound (US)": 0.45359237,
            "gram": 0.001,
        },
    },
    "Height": {
        "ancestor": 3036277,
        "lab_key": "height",
        "low": 100,
        "high": 250,
        "drop_units": [
            "gram",
            "kilogram",
            "ounce (avoirdupois)",
            "pound (US)",
        ],
        "unit_map": {
            "inch (international)": 2.54,
            "foot (international)": 30.48,
            "meter": 100.0,
            "No matching concept": 2.54,
        },
    },
}


In [ ]:
def pull_and_clean_measurement(label, cfg):
    """Query one measurement family and apply its original QC rules."""
    raw = get_lab_by_ancestor_all(
        cfg["ancestor"]
    )

    clean = clean_and_filter_units(
        raw,
        lab_name=label,
        drop_units=cfg.get("drop_units"),
        low=cfg.get("low"),
        high=cfg.get("high"),
        unit_map=cfg.get("unit_map"),
        drop_outliers=cfg.get(
            "drop_outliers",
            False,
        ),
    )

    print(
        f"{label}: {len(raw):,} raw -> "
        f"{len(clean):,} cleaned rows"
    )

    return clean


# Pull each additional measurement family once.
additional_measurement_data = {
    label: pull_and_clean_measurement(
        label,
        cfg,
    )
    for label, cfg in ADDITIONAL_MEASUREMENTS.items()
}

body_measurement_data = {
    label: pull_and_clean_measurement(
        label,
        cfg,
    )
    for label, cfg in BODY_MEASUREMENTS.items()
}


In [ ]:
def build_measurement_block(
    cutoff_df,
    measurement_data,
    configs,
    specs,
):
    """Summarize a collection of measurement families at one landmark."""
    out = (
        cutoff_df[["person_id"]]
        .drop_duplicates()
        .copy()
    )

    for label, cfg in configs.items():
        feats = build_single_lab_features_at_cutoff(
            cutoff_df=cutoff_df,
            lab_df=measurement_data[label],
            lab_key=cfg["lab_key"],
            lab_specs=specs,
            lab_label_prefix=label,
            lab_date_col="measurement_datetime",
            lab_value_col="value_as_number",
        )

        out = out.merge(
            feats,
            on="person_id",
            how="left",
            validate="one_to_one",
        )

    return out


additional_lab_blocks = {}
body_blocks = {}

for name, cohort in landmark_cohorts.items():
    print(f"\nBuilding additional measurements: {name}")

    labs = build_measurement_block(
        cohort,
        additional_measurement_data,
        ADDITIONAL_MEASUREMENTS,
        LAB_SPECS,
    )

    body = build_measurement_block(
        cohort,
        body_measurement_data,
        BODY_MEASUREMENTS,
        BODY_SPECS,
    )

    additional_lab_blocks[name] = labs
    body_blocks[name] = body

    save_gcs_parquet(
        labs,
        f"{CURATED_DIR}/additional_labs_vitals_{name}.parquet",
    )

    save_gcs_parquet(
        body,
        f"{CURATED_DIR}/body_measurements_{name}.parquet",
    )


## Medication Exposure

Medication features capture treatment exposure available before each landmark. For each medication class, the block records prior exposure and recency of the most recent pre-landmark exposure.


In [ ]:
from google.cloud import bigquery
import pandas as pd
import os

from google.cloud import bigquery
import pandas as pd
import os

def get_drug_by_ancestor_all(ancestor_id):
    """
    Query drug exposures for all participants in AoU by ancestor concept_id.
    Returns detailed drug exposure rows with concept metadata.
    
    Parameters
    ----------
    ancestor_id : int
        OMOP concept_id for drug ancestor (e.g., Metformin = 1503297)
    """

    client = bq_client

    # --- Get descendants ---
    sql_desc = f"""
    SELECT descendant_concept_id
    FROM `{CDR_DATASET}.concept_ancestor`
    WHERE ancestor_concept_id = @anc_id
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("anc_id", "INT64", int(ancestor_id))]
    )
    desc = client.query(sql_desc, job_config=job_config).to_dataframe()["descendant_concept_id"].tolist()

    # --- Query drug_exposure + concept details ---
    sql = f"""
    SELECT
      d.person_id,
      d.drug_exposure_id,
      d.drug_concept_id,
      d.drug_exposure_start_date,
      d.drug_exposure_end_date,
      d.days_supply,
      d.quantity,
      d.drug_type_concept_id,
      dtype.concept_name AS drug_type_name,
      
      -- Standard concept info
      c.concept_name     AS drug_concept_name,
      c.vocabulary_id    AS vocabulary,
      c.concept_class_id AS concept_class,
      c.standard_concept AS is_standard,
      c.valid_start_date,
      c.valid_end_date,
      c.invalid_reason,
      
      -- Route info
      d.route_concept_id,
      r.concept_name     AS route_concept_name,
      
      -- Source values (raw strings from EHR/claims)
      d.drug_source_value,
      d.dose_unit_source_value,
      d.route_source_value
    FROM `{CDR_DATASET}.drug_exposure` d
    JOIN `{CDR_DATASET}.concept` c
      ON d.drug_concept_id = c.concept_id
    LEFT JOIN `{CDR_DATASET}.concept` r
      ON d.route_concept_id = r.concept_id
    LEFT JOIN `{CDR_DATASET}.concept` dtype
      ON d.drug_type_concept_id = dtype.concept_id
    WHERE d.drug_concept_id IN UNNEST(@desc_ids)
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("desc_ids", "INT64", desc)]
    )
    df = client.query(sql, job_config=job_config).to_dataframe()

    # Convert datetimes
    df["drug_exposure_start_date"] = pd.to_datetime(df["drug_exposure_start_date"], errors="coerce", utc=True)
    df["drug_exposure_end_date"] = pd.to_datetime(df["drug_exposure_end_date"], errors="coerce", utc=True)

    return df


In [ ]:
antidiabetic_ancestors = {
    "metformin": [1503297],
    "sulfonylureas": [
        1502855, 1594973, 1530014, 1502809,
        1560171, 1559684, 19059796, 1597756
    ],
    "thiazolidinediones": [
        1525215, 1547504
    ],
    "dpp4i": [
        1580747, 40166035, 40239216, 43013884, 19122137
    ],
    "glp1ra": [
        1583722, 40170911, 45774435, 793143, 44506754, 44816332
    ],
    "sglt2i": [
        45774751, 44785829, 43526465, 793293, 1201369, 43009020
    ],
    "insulin": [
        1544838, 35605670, 35602717, 1516796, 1502095, 46221581, 1560023,
        36851361, 35198096, 12351304, 36863356, 36863903, 36859783, 36861186,
        1567198, 36858942, 36862515, 36863468, 36862049, 35884381, 1531601,
        1568898, 1513878, 36859435, 19013951, 1590165, 1569987, 1586346,
        19090204, 1513843, 1513849, 1562658, 19090226, 19090221, 1586389,
        19090244, 19090247, 19090249, 19090250, 19090251, 19013926,
        19090261, 19090187
    ]
}


In [ ]:
htn_ancestors = {
    "htn_acei": [
        19040051, 43009074, 36878917, 43009010, 19102107, 19122327, 1310756, 19050216,
        1342439, 1363749, 1331235, 1373225, 1335471, 1340128, 1341927, 1334456, 1308216,
        36850184, 36860865, 36851966, 36851676, 36862930, 36862785, 45775374, 45775529,
        45775544, 45775375, 1342001, 45775539, 45775527
    ],
    "htn_arb": [
        36851542, 40235485, 1346686, 40226742, 1317640, 1351557, 1347384, 1367500, 1308842,
        36862792, 35884389, 36854445
    ],
    "htn_thiazides": [
        19010015, 1376289, 948787, 904639, 19099181, 905273, 907013, 19010493, 1316354,
        992590, 978555, 1395058, 974166
    ],
    "htn_loops": [
        19025498, 932745, 942350, 1254409, 956874
    ],
    "htn_potassium_sparing": [
        991382, 904542
    ],
    "htn_mra": [
        970250, 1309799, 1537451
    ],
    "htn_beta_blockers": [
        36856962, 19100451, 19081284, 1327978, 19024904, 19063575, 950370, 19049145, 1322081,
        1345858, 1313200, 1319998, 1314577, 1386957, 1370109, 1346823, 1353766, 902427,
        1338005, 1314002, 1355703, 1307046
    ],
    "htn_ccb_non_dhp": [
        1307863, 1328165, 19080432
    ],
    "htn_ccb_dhp": [
        43008998, 19102106, 19089969, 36851681, 19071995, 1326012, 19004539, 43009040,
        1319133, 19020061, 1318137, 19015802, 1353776, 1318853, 1332418, 35197931,
        36855130, 36855016, 36853980, 36855657, 36863734, 36852719, 36855160, 1319880,
        36862614, 36851015, 36863745, 36863753
    ],
    "htn_direct_renin": [
        1317967
    ]
}


In [ ]:
lipid_ancestors = {
    "lipid_statins": [
        40165636, 1549686, 1592085, 1551860, 36852340,
        1510813, 1539403, 1545958, 36849752
    ],
    "lipid_ezetimibe": [
        1526475
    ],
    "lipid_pcsk9i": [
        46275447, 46287466
    ],
    "lipid_bile_acid_seq": [
        19095309, 1501617, 1518148
    ],
    "lipid_fibrates": [
        19022956, 1558242, 1551803, 43560137, 35197926,
        36850481, 19060156
    ],
    "lipid_niacin": [
        1517824
    ]
}


In [ ]:
import gc
import pandas as pd
import numpy as np

# ==========================================================
# 0) Helper: build group dfs from ancestor lists (memory safe-ish)
# ==========================================================
def build_drug_group_dfs(ancestor_dict, label):
    """
    Returns: dict[group_name] -> concatenated drug_exposure df
    """
    out = {}
    for group, anc_ids in ancestor_dict.items():
        print(f"\n💊 [{label}] Building group: {group} (n_ancestors={len(anc_ids)})")
        group_dfs = []
        for anc in anc_ids:
            df = get_drug_by_ancestor_all(int(anc))
            group_dfs.append(df)
        out[group] = pd.concat(group_dfs, ignore_index=True) if group_dfs else pd.DataFrame()
        print(f"   ✅ rows: {len(out[group]):,}")
        del group_dfs
        gc.collect()
    return out


# ==========================================================
# 1) Exposure summarizer (ever + recency days)
# ==========================================================
def summarize_single_med_ever_recency(
    med_df: pd.DataFrame,
    med_label: str,
    cohort_df: pd.DataFrame,
    *,
    id_col: str = "person_id",
    start_col: str = "drug_exposure_start_date",
    cutoff_date_col: str = "cutoff_date",
) -> pd.DataFrame:
    """
    For each person in cohort_df:
      - {med_label}_ever: any exposure start_date < cutoff_date
      - {med_label}_recency_days: (cutoff_date - last_start_date).days for last exposure before cutoff
    """

    # Base: one row per person with cutoff_date
    base = cohort_df[[id_col, cutoff_date_col]].copy()
    base[cutoff_date_col] = pd.to_datetime(base[cutoff_date_col], errors="coerce", utc=True)

    # Quick exits
    if med_df is None or len(med_df) == 0:
        out = base[[id_col]].copy()
        out[f"{med_label}_ever"] = 0
        out[f"{med_label}_recency_days"] = pd.Series([pd.NA] * len(out), dtype="Int64")
        return out

    # Keep only needed cols from meds, coerce types
    m = med_df[[id_col, start_col]].copy()
    m[start_col] = pd.to_datetime(m[start_col], errors="coerce", utc=True)

    # Merge cutoff into medication rows and keep pre-cutoff exposures
    mm = m.merge(base, on=id_col, how="inner")
    mm = mm.loc[mm[start_col].notna() & mm[cutoff_date_col].notna()]
    mm = mm.loc[mm[start_col] < mm[cutoff_date_col]]

    if mm.empty:
        out = base[[id_col]].copy()
        out[f"{med_label}_ever"] = 0
        out[f"{med_label}_recency_days"] = pd.Series([pd.NA] * len(out), dtype="Int64")
        return out

    # last pre-cutoff exposure per person
    last_start = (
        mm.sort_values([id_col, start_col])
          .groupby(id_col, as_index=False)
          .tail(1)[[id_col, start_col, cutoff_date_col]]
          .rename(columns={start_col: "last_start_date"})
    )

    out = base[[id_col]].merge(last_start[[id_col, "last_start_date", cutoff_date_col]], on=id_col, how="left")
    out[f"{med_label}_ever"] = out["last_start_date"].notna().astype("int8")

    rec = (out[cutoff_date_col] - out["last_start_date"]).dt.days
    out[f"{med_label}_recency_days"] = pd.to_numeric(rec, errors="coerce").astype("Int64")

    out = out.drop(columns=["last_start_date", cutoff_date_col])
    return out

# ==========================================================
# 2) Build exposure block for one cutoff df
# ==========================================================
def build_med_exposure_block_for_cutoff(
    cutoff_df: pd.DataFrame,
    cutoff_label: str,
    *,
    diab_med_dfs: dict,
    htn_med_dfs: dict,
    lipid_med_dfs: dict,
    id_col: str = "person_id",
    cutoff_date_col: str = "cutoff_date",   # <-- set this to your actual column in df_cutoff_6m, etc.
    start_col: str = "drug_exposure_start_date",
    make_recent_30d: bool = True,
    suffix_outputs: bool = False,
) -> pd.DataFrame:
    """
    Returns one df with all DM + HTN + lipid group features for this cutoff window.

    If suffix_outputs=True, appends _{cutoff_label} to the feature columns.
    """
    if cutoff_date_col not in cutoff_df.columns:
        raise KeyError(
            f"cutoff_df is missing '{cutoff_date_col}'. "
            f"Available columns: {list(cutoff_df.columns)[:40]}"
        )

    cut = cutoff_df[[id_col, cutoff_date_col]].copy()

    # --------------------------
    # DM groups
    # --------------------------
    diab_flags = cut[[id_col]].copy()
    for group, df in diab_med_dfs.items():
        summ = summarize_single_med_ever_recency(
            med_df=df,
            med_label=group,
            cohort_df=cut,
            id_col=id_col,
            start_col=start_col,
            cutoff_date_col=cutoff_date_col,
        )
        diab_flags = diab_flags.merge(summ, on=id_col, how="left")

    # --------------------------
    # HTN groups
    # --------------------------
    htn_flags = cut[[id_col]].copy()
    for group, df in htn_med_dfs.items():
        summ = summarize_single_med_ever_recency(
            med_df=df,
            med_label=group,
            cohort_df=cut,
            id_col=id_col,
            start_col=start_col,
            cutoff_date_col=cutoff_date_col,
        )
        htn_flags = htn_flags.merge(summ, on=id_col, how="left")

    # --------------------------
    # Lipid groups
    # --------------------------
    lipid_flags = cut[[id_col]].copy()
    for group, df in lipid_med_dfs.items():
        summ = summarize_single_med_ever_recency(
            med_df=df,
            med_label=group,
            cohort_df=cut,
            id_col=id_col,
            start_col=start_col,
            cutoff_date_col=cutoff_date_col,
        )
        lipid_flags = lipid_flags.merge(summ, on=id_col, how="left")

    # Merge blocks
    all_meds = (
        diab_flags
        .merge(htn_flags, on=id_col, how="outer")
        .merge(lipid_flags, on=id_col, how="outer")
    )

    # Fill ever flags
    ever_cols = [c for c in all_meds.columns if c.endswith("_ever")]
    all_meds[ever_cols] = all_meds[ever_cols].fillna(0).astype("int8")

    # Optionally create recent_30d flags then drop recency_days
    recency_cols = [c for c in all_meds.columns if c.endswith("_recency_days")]
    if make_recent_30d:
        for c in recency_cols:
            base = c.replace("_recency_days", "")
            days = pd.to_numeric(all_meds[c], errors="coerce")
            all_meds[f"{base}_recent_30d"] = days.le(30).fillna(False).astype("int8")

    # If you want to keep raw recency_days, comment this out
    all_meds = all_meds.drop(columns=recency_cols, errors="ignore")

    # Optional suffixing
    if suffix_outputs:
        rename = {c: f"{c}_{cutoff_label}" for c in all_meds.columns if c != id_col}
        all_meds = all_meds.rename(columns=rename)

    return all_meds


In [ ]:
# Build medication class source tables once.
diab_med_dfs = build_drug_group_dfs(
    antidiabetic_ancestors,
    label="DM",
)
htn_med_dfs = build_drug_group_dfs(
    htn_ancestors,
    label="HTN",
)
lipid_med_dfs = build_drug_group_dfs(
    lipid_ancestors,
    label="Lipid",
)

medication_blocks = {}

for name, cohort in landmark_cohorts.items():
    print(f"\nBuilding medication features: {name}")

    block = build_med_exposure_block_for_cutoff(
        cutoff_df=cohort,
        cutoff_label=name,
        diab_med_dfs=diab_med_dfs,
        htn_med_dfs=htn_med_dfs,
        lipid_med_dfs=lipid_med_dfs,
        cutoff_date_col="cutoff_date",
    )

    ensure_person_level(
        block,
        f"medications_{name}",
    )

    medication_blocks[name] = block

    save_gcs_parquet(
        block,
        f"{CURATED_DIR}/medications_{name}.parquet",
    )


## Survey Features

Survey-derived predictors are restricted to responses completed strictly before each participant's landmark.

For each survey domain, the latest eligible pre-landmark response is selected. This rule is applied to the Basics, Lifestyle, Social Determinants of Health, and Personal and Family Health History surveys.


In [ ]:
aou_basic_surveys = read_gcs_csv(
    INPUT_PATHS["basic_surveys"]
)
aou_social_health_surveys = read_gcs_csv(
    INPUT_PATHS["social_surveys"]
)
lifestyle_surveys = read_gcs_csv(
    INPUT_PATHS["lifestyle_surveys"]
)
fh_df = read_gcs_csv(
    INPUT_PATHS["family_history_surveys"]
)


In [ ]:
# ==========================================================
# SURVEY FEATURE POLICY (UNIFIED, REUSABLE)
# ==========================================================
# Policy we will enforce for ALL survey-derived blocks:
#   ✅ Keep the encoded value for each concept
#   ✅ Keep ONE missing flag per concept: <concept>_missing
#   ✅ Drop skip/pna/dont_know flags (avoid feature explosion + SHAP fragmentation)
#   ✅ Add ONE "has survey" flag per block (per landmark):
#         has_<block>_survey  = 1 if the person had an eligible pre-cutoff survey row
#   ✅ Always return ONE ROW PER COHORT PERSON (left-join backbone)
#
# Notes:
#   - XGB handles NA natively; missing flags capture participation/data-availability signal cleanly.
#   - Logistic baseline benefits hugely from removing correlated response-behavior flags.
# ==========================================================

import pandas as pd
import numpy as np
import re
import ast


# ==========================================================
# 0) Canonical token cleaning
# ==========================================================
def _clean_token(x):
    """
    Normalize survey responses to stable tokens:
      - pd.NA for true missing
      - lower_snake_case tokens for values
      - special tokens: 'skip', 'pna', 'dont_know'
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().lower()
    s = re.sub(r"\s+", " ", s)

    # canonical special responses
    s = s.replace("prefer not to answer", "pna")
    s = s.replace("don't know", "dont_know").replace("dont know", "dont_know")

    # normalize punctuation/separators
    s = s.replace("/", "_").replace("-", "_")
    s = re.sub(r"[^a-z0-9_ ]", "", s)
    s = s.replace(" ", "_")
    s = re.sub(r"_+", "_", s).strip("_")

    if s in {"", "nan", "none"}:
        return pd.NA

    return s


def _std(series: pd.Series) -> pd.Series:
    """Return cleaned tokens as pandas 'string' dtype."""
    return series.apply(_clean_token).astype("string")


# ==========================================================
# 1) Policy helpers
# ==========================================================
def _concept_missing_flag_from_raw(df: pd.DataFrame, raw_series: pd.Series, concept: str) -> None:
    """
    Create <concept>_missing based on:
      - true NA
      - or explicit response tokens: skip/pna/dont_know
    We treat all of these as missing for modeling.
    """
    s = _std(raw_series)
    df[f"{concept}_missing"] = (s.isna() | s.isin(["skip", "pna", "dont_know"])).astype("Int64")


def _encode_yes_no_policy(raw_series: pd.Series) -> pd.Series:
    """
    yes/no encoding under policy:
      - yes -> 1, no -> 0
      - skip/pna/dont_know/missing -> NA
    """
    s = _std(raw_series)
    s = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)
    return s.map({"yes": 1, "no": 0}).astype("Int64")


def _encode_ordered_policy(raw_series: pd.Series, mapping: dict) -> pd.Series:
    """
    ordinal encoding under policy:
      - map tokens to Int64
      - skip/pna/dont_know/missing -> NA
      - unknown tokens -> NA
    """
    s = _std(raw_series)
    s = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)
    return s.map(mapping).astype("Int64")


def _split_numeric_policy(df: pd.DataFrame, raw_series: pd.Series, concept: str) -> None:
    """
    Numeric + response-mix columns:
      - <concept>_num: Float64 numeric value
      - <concept>_missing: (missing/skip/pna/dont_know) flag
    We intentionally DROP code expansion to keep the feature set compact.
    """
    s = _std(raw_series)
    df[f"{concept}_missing"] = (s.isna() | s.isin(["skip", "pna", "dont_know"])).astype("Int64")
    s_num = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)
    df[f"{concept}_num"] = pd.to_numeric(s_num, errors="coerce").astype("Float64")


def _parse_multiselect_list(raw_series: pd.Series) -> pd.Series:
    """
    Normalize multi-select cells into list of cleaned tokens.
    """
    def _parse_one(x):
        if pd.isna(x):
            return []
        if isinstance(x, (list, tuple)):
            raw_list = list(x)
        else:
            s = str(x).strip()
            if s == "":
                return []
            if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
                try:
                    val = ast.literal_eval(s)
                    raw_list = list(val) if isinstance(val, (list, tuple)) else [s]
                except Exception:
                    raw_list = [s]
            elif "|" in s:
                raw_list = [t.strip() for t in s.split("|") if t.strip()]
            elif "," in s:
                raw_list = [t.strip() for t in s.split(",") if t.strip()]
            else:
                raw_list = [s]

        cleaned = []
        for v in raw_list:
            tok = _clean_token(v)
            if tok is pd.NA:
                continue
            cleaned.append(str(tok))
        return cleaned

    return raw_series.apply(_parse_one)


def drop_response_behavior_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Hard guard: drop any leftover *_skip/_pna/_dont_know columns if they exist.
    """
    drop = [c for c in df.columns if c.endswith("_skip") or c.endswith("_pna") or c.endswith("_dont_know")]
    if drop:
        df = df.drop(columns=drop)
    return df


# ==========================================================
# 2) Cohort backbone + pre-cutoff selection (generic)
# ==========================================================
def select_latest_pre_cutoff_one_row(
    survey_df: pd.DataFrame,
    cutoff_df: pd.DataFrame,
    id_col: str,
    cutoff_col: str,
    date_col: str,
    max_lookback_days: int | None = None,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Returns:
      - selected rows (only for people with eligible pre-cutoff surveys)
      - has_survey series aligned to cutoff_df backbone (1 if eligible row exists else 0)
    """
    cohort = cutoff_df[[id_col, cutoff_col]].drop_duplicates().copy()
    backbone = cohort[[id_col]].drop_duplicates().copy()

    tmp = survey_df.merge(cohort, on=id_col, how="inner").copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce", utc=True)
    tmp[cutoff_col] = pd.to_datetime(tmp[cutoff_col], errors="coerce", utc=True)

    tmp = tmp.loc[tmp[date_col].notna() & tmp[cutoff_col].notna()]
    tmp = tmp.loc[tmp[date_col] < tmp[cutoff_col]]

    if max_lookback_days is not None:
        delta = (tmp[cutoff_col] - tmp[date_col]).dt.days
        tmp = tmp.loc[delta <= max_lookback_days]

    sel = (
        tmp.sort_values([id_col, date_col])
           .groupby(id_col, as_index=False)
           .tail(1)
           .drop(columns=[cutoff_col])
    )

    has = backbone.merge(sel[[id_col]].assign(_has=1), on=id_col, how="left")["_has"].fillna(0).astype("Int64")
    return sel, has


# ==========================================================
# 3) DEMOGRAPHICS (BASIC SURVEYS) - STATIC / NOT CUTOFF-ANCHORED
# ==========================================================
# We treat these as static attributes for your pipeline:
#   - sex_at_birth, race/ethnicity, education, income, employment, insurance, housing, marital, active duty
#
# Implementation:
#   - take latest available response per person (no cutoff)
#   - left-join to cohort backbone (so full cohort)
#   - apply policy: encoded value + one missing flag, no skip/pna/dk flags
# ==========================================================

def build_demographics_basic_static(
    aou_basic_surveys: pd.DataFrame,
    cohort_df: pd.DataFrame,
    id_col: str = "person_id",
    survey_date_col: str = "basics_survey_date",
) -> pd.DataFrame:
    """
    Output:
      - one row per person in cohort_df
      - compact demographics encodings (no one-hot explosion)
      - per-concept missing flags
      - has_basic_survey flag (did they have ANY survey row)
    """
    backbone = cohort_df[[id_col]].drop_duplicates().copy()

    df = aou_basic_surveys.copy()
    if survey_date_col in df.columns:
        df[survey_date_col] = pd.to_datetime(df[survey_date_col], errors="coerce", utc=True)
        df = df.sort_values([id_col, survey_date_col]).groupby(id_col, as_index=False).tail(1)
    else:
        df = df.drop_duplicates(subset=[id_col], keep="last")

    # keep only cohort
    df = backbone.merge(df, on=id_col, how="left")

    out = df[[id_col]].copy()
    out["has_basic_survey"] = df[id_col].isin(aou_basic_surveys[id_col].unique()).astype("Int64")

    # sex
    if "Biological_Sex_At_Birth_Sex_At_Birth" in df.columns:
        _concept_missing_flag_from_raw(out, df["Biological_Sex_At_Birth_Sex_At_Birth"], "sex_male")
        out["sex_male"] = _encode_yes_no_policy(df["Biological_Sex_At_Birth_Sex_At_Birth"]).replace({1: 1, 0: 0})

        # AoU uses male/female not yes/no in this column
        s = _std(df["Biological_Sex_At_Birth_Sex_At_Birth"])
        s = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)
        out["sex_male"] = s.map({"male": 1, "female": 0}).astype("Int64")

    # hispanic any
    if "Hispanic_Hispanic_Specific" in df.columns:
        _concept_missing_flag_from_raw(out, df["Hispanic_Hispanic_Specific"], "hispanic_any")
        s = _std(df["Hispanic_Hispanic_Specific"])
        s2 = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)
        out["hispanic_any"] = (~s2.isna() & ~s2.isin(["no", "not_hispanic"])).astype("Int64")

    # race (keep as compact categorical code)
    if "Race_What_Race_Ethnicity" in df.columns:
        s = _std(df["Race_What_Race_Ethnicity"])
        out["race_any_missing"] = (s.isna() | s.isin(["skip", "pna", "dont_know"])).astype("Int64")
        # keep as token (string) for now; optionally map to small category set later
        out["race_any"] = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)

    # housing unstable (yes/no)
    if "Living_Situation_Stable_House_Concern" in df.columns:
        _concept_missing_flag_from_raw(out, df["Living_Situation_Stable_House_Concern"], "housing_unstable")
        s = _std(df["Living_Situation_Stable_House_Concern"])
        s2 = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)
        out["housing_unstable"] = s2.map({"yes": 1, "no": 0}).astype("Int64")

    # ordinal encodings
    edu_map = {
        "never_attended": 0,
        "one_through_four": 1,
        "five_through_eight": 2,
        "nine_through_eleven": 3,
        "twelve_or_ged": 4,
        "college_one_to_three": 5,
        "college_graduate": 6,
        "advanced_degree": 7,
    }
    if "Education_Level_Highest_Grade" in df.columns:
        _concept_missing_flag_from_raw(out, df["Education_Level_Highest_Grade"], "education_level")
        out["education_level"] = _encode_ordered_policy(df["Education_Level_Highest_Grade"], edu_map)

    income_map = {
        "less_10k": 1,
        "10k_25k": 2,
        "25k_35k": 3,
        "35k_50k": 4,
        "50k_75k": 5,
        "75k_100k": 6,
        "100k_150k": 7,
        "150k_200k": 8,
        "more_200k": 9,
    }
    if "Income_Annual_Income" in df.columns:
        _concept_missing_flag_from_raw(out, df["Income_Annual_Income"], "income_level")
        out["income_level"] = _encode_ordered_policy(df["Income_Annual_Income"], income_map)

    # compact categoricals: keep as tokens (no one-hot here)
    compact_cats = {
        "Employment_Employment_Status": "employment_status",
        "Insurance_Health_Insurance": "insurance_status",
        "Health_Insurance_Health_Insurance_Type": "insurance_type",
        "Home_Own_Current_Home_Own": "home_own",
        "Living_Situation_Current_Living": "living_situation",
        "Marital_Status_Current_Marital_Status": "marital_status",
        "Active_Duty_Active_Duty_Serve_Status": "active_duty",
    }
    for raw, concept in compact_cats.items():
        if raw in df.columns:
            s = _std(df[raw])
            out[f"{concept}_missing"] = (s.isna() | s.isin(["skip", "pna", "dont_know"])).astype("Int64")
            out[concept] = s.where(~s.isin(["skip", "pna", "dont_know"]), pd.NA)

    out = drop_response_behavior_flags(out)

    # optional: one-hot only for these compact categoricals (controlled)
    one_hot_cols = [c for c in compact_cats.values() if c in out.columns]
    if one_hot_cols:
        out = pd.get_dummies(out, columns=one_hot_cols, drop_first=False, dtype="Int64")

    # avoid dummies for raw race tokens unless you want them; safe to dummy since race set is small
    if "race_any" in out.columns:
        out = pd.get_dummies(out, columns=["race_any"], drop_first=False, dtype="Int64")

    out = out.loc[:, ~out.columns.duplicated(keep="first")].copy()
    return out


# ==========================================================
# 4) LIFESTYLE - LANDMARK-AWARE + COMPACT POLICY
# ==========================================================
def prepare_lifestyle_compact_for_cutoff(
    lifestyle_surveys: pd.DataFrame,
    cutoff_df: pd.DataFrame,
    id_col: str = "person_id",
    cutoff_col: str = "cutoff_date",
    date_col: str = "lifestyle_survey_date",
    max_lookback_days: int | None = 730,
) -> pd.DataFrame:
    """
    Output:
      - one row per person in cutoff_df (cohort preserved)
      - has_lifestyle_survey (eligible pre-cutoff row)
      - compact encodings + one missing flag per concept
      - no skip/pna/dont_know columns
    """
    backbone = cutoff_df[[id_col]].drop_duplicates().copy()

    sel, has = select_latest_pre_cutoff_one_row(
        lifestyle_surveys,
        cutoff_df,
        id_col=id_col,
        cutoff_col=cutoff_col,
        date_col=date_col,
        max_lookback_days=max_lookback_days,
    )

    base = backbone.merge(sel, on=id_col, how="left")
    out = base[[id_col]].copy()
    out["has_lifestyle_survey"] = has.values

    # Alcohol participation
    if "Alcohol_Alcohol_Participant" in base.columns:
        _concept_missing_flag_from_raw(out, base["Alcohol_Alcohol_Participant"], "alcohol_participant")
        out["alcohol_participant"] = _encode_yes_no_policy(base["Alcohol_Alcohol_Participant"])

    # Alcohol binge occurrence
    if "Alcohol_6_or_More_Drinks_Occurrence" in base.columns:
        mapping_binge = {
            "never_in_last_year": 0,
            "less_than_monthly": 1,
            "monthly": 2,
            "weekly": 3,
            "daily": 4,
        }
        _concept_missing_flag_from_raw(out, base["Alcohol_6_or_More_Drinks_Occurrence"], "alcohol_binge_freq")
        out["alcohol_binge_freq"] = _encode_ordered_policy(base["Alcohol_6_or_More_Drinks_Occurrence"], mapping_binge)

    # Alcohol drinking frequency
    if "Alcohol_Drink_Frequency_Past_Year" in base.columns:
        mapping_alc_freq = {
            "never": 0,
            "monthly_or_less": 1,
            "2_to_4_per_month": 2,
            "2_to_3_per_week": 3,
            "4_or_more_per_week": 4,
        }
        _concept_missing_flag_from_raw(out, base["Alcohol_Drink_Frequency_Past_Year"], "alcohol_drink_freq")
        out["alcohol_drink_freq"] = _encode_ordered_policy(base["Alcohol_Drink_Frequency_Past_Year"], mapping_alc_freq)

    # Alcohol daily quantity
    if "Alcohol_Average_Daily_Drink_Count" in base.columns:
        mapping_alc_qty = {"1_or_2": 1, "3_or_4": 2, "5_or_6": 3, "7_to_9": 4, "10_or_more": 5}
        _concept_missing_flag_from_raw(out, base["Alcohol_Average_Daily_Drink_Count"], "alcohol_daily_qty")
        out["alcohol_daily_qty"] = _encode_ordered_policy(base["Alcohol_Average_Daily_Drink_Count"], mapping_alc_qty)

    # Tobacco participation yes/no
    participant_like = {
        "Cigar_Smoking_Cigar_Smoke_Participant": "cigar_participant",
        "Electronic_Smoking_Electric_Smoke_Participant": "vape_participant",
        "Hookah_Smoking_Hookah_Smoke_Participant": "hookah_participant",
        "Smokeless_Tobacco_Smokeless_Tobacco_Participant": "smokeless_participant",
        "Smoking_100_Cigs_Lifetime": "cigs_100_lifetime",
    }
    for raw, concept in participant_like.items():
        if raw in base.columns:
            _concept_missing_flag_from_raw(out, base[raw], concept)
            out[concept] = _encode_yes_no_policy(base[raw])

    # Tobacco frequencies
    freq_map = {"not_at_all": 0, "some_days": 1, "every_day": 2}
    freq_cols = {
        "Cigar_Smoking_Current_Cigar_Frequency": "cigar_freq",
        "Electronic_Smoking_Electric_Smoke_Frequency": "vape_freq",
        "Hookah_Smoking_Current_Hookah_Frequency": "hookah_freq",
        "Smokeless_Tobacco_Smokeless_Tobacco_Frequency": "smokeless_freq",
        "Smoking_Smoke_Frequency": "cigarette_freq",
    }
    for raw, concept in freq_cols.items():
        if raw in base.columns:
            _concept_missing_flag_from_raw(out, base[raw], concept)
            out[concept] = _encode_ordered_policy(base[raw], freq_map)

    # Mixed numeric (compact: only *_num + *_missing)
    numeric_mixed = {
        "Attempt_Quit_Smoking_Completely_Quit_Age": "quit_age",
        "Smoking_Average_Daily_Cigarette_Number": "cigs_avg_daily",
        "Smoking_Current_Daily_Cigarette_Number": "cigs_current_daily",
        "Smoking_Daily_Smoke_Starting_Age": "smoke_start_age",
        "Smoking_Number_Of_Years": "smoke_years",
    }
    for raw, concept in numeric_mixed.items():
        if raw in base.columns:
            _split_numeric_policy(out, base[raw], concept)

    # Quit attempt
    if "Smoking_Serious_Quit_Attempt" in base.columns:
        mapping_quit_attempt = {"attempt_quit_smoking": 1, "no_attempt_quit_smoking": 0}
        _concept_missing_flag_from_raw(out, base["Smoking_Serious_Quit_Attempt"], "serious_quit_attempt")
        out["serious_quit_attempt"] = _encode_ordered_policy(base["Smoking_Serious_Quit_Attempt"], mapping_quit_attempt)

    # Marijuana
    if "Past_3_Month_Use_Frequency_Marijuana_3_Month_Use" in base.columns:
        mapping_mj = {"never": 0, "one_or_twice": 1, "once_or_twice": 1, "monthly": 2, "weekly": 3, "daily": 4}
        _concept_missing_flag_from_raw(out, base["Past_3_Month_Use_Frequency_Marijuana_3_Month_Use"], "marijuana_freq")
        out["marijuana_freq"] = _encode_ordered_policy(base["Past_3_Month_Use_Frequency_Marijuana_3_Month_Use"], mapping_mj)

    # Recreational drug list -> compact flags
    if "Recreational_Drug_Use_Which_Drugs_Used" in base.columns:
        lists = _parse_multiselect_list(base["Recreational_Drug_Use_Which_Drugs_Used"])
        bad = {"none_of_these_drugs", "skip", "pna", "dont_know", ""}

        out["rec_any_drug"] = lists.apply(lambda lst: int(any(str(v).strip() not in bad for v in lst))).astype("Int64")
        out["rec_any_drug_missing"] = base["Recreational_Drug_Use_Which_Drugs_Used"].isna().astype("Int64")

    out = drop_response_behavior_flags(out)
    out = out.loc[:, ~out.columns.duplicated(keep="first")].copy()
    return out


# ==========================================================
# 5) SDoH / SOCIAL HEALTH - LANDMARK-AWARE HYBRID + COMPACT POLICY
# ==========================================================
def prepare_sdoh_compact_for_cutoff(
    aou_social_health_surveys: pd.DataFrame,
    cutoff_df: pd.DataFrame,
    id_col: str = "person_id",
    cutoff_col: str = "cutoff_date",
    date_col: str = "social_health_survey_date",
    max_lookback_days: int | None = None,
) -> pd.DataFrame:
    """
    Compact SDoH design:
      - One row per cohort person
      - has_sdoh_survey flag (eligible pre-cutoff row)
      - Per-question ordinal encodings + <concept>_missing
      - Composite scores + composite missing flags if desired
      - NO skip/pna/dont_know flags
    """
    backbone = cutoff_df[[id_col]].drop_duplicates().copy()

    if date_col not in aou_social_health_surveys.columns:
        raise KeyError(
            f"SDoH survey date column '{date_col}' is required "
            "for landmark-safe feature construction."
        )

    sel, has = select_latest_pre_cutoff_one_row(
        aou_social_health_surveys,
        cutoff_df,
        id_col=id_col,
        cutoff_col=cutoff_col,
        date_col=date_col,
        max_lookback_days=max_lookback_days,
    )

    base = backbone.merge(sel, on=id_col, how="left")
    out = base[[id_col]].copy()
    out["has_sdoh_survey"] = has.values

    # mappings
    agree_map = {
        "strongly_disagree": 1,
        "disagree": 2,
        "neutral_neither_agree_nor_disagree": 3,
        "neutral": 3,
        "agree": 4,
        "strongly_agree": 5,
    }
    freq_map = {
        "never": 1,
        "less_than_once_a_year": 2,
        "a_few_times_a_year": 3,
        "a_few_times_a_month": 4,
        "at_least_once_a_week": 5,
        "almost_everyday": 6,
    }
    lonely_map = {"never": 1, "rarely": 2, "sometimes": 3, "often": 4}
    stress_map = {"never": 1, "almost_never": 2, "sometimes": 3, "fairly_often": 4, "very_often": 5}

    # raw -> concept maps (same as your earlier block, but compact)
    loneliness_raw = {
        "How_often_do_you_feel_isolated_from_others?": "lonely_isolated",
        "How_often_do_you_feel_lack_companionship?": "lonely_lack_companionship",
        "How_often_do_you_feel_left_out?": "lonely_left_out",
        "How_often_do_you_feel_that_people_are_around_you_but_not_with_you?": "lonely_withyou",
        "How_often_do_you_feel_that_there_is_no_one_you_can_turn_to?": "lonely_alone",
    }
    harassment_raw = {
        "In_your_day-to-day_life,_how_often_are_you_called_names_or_insulted?": "harass_insults",
        "In_your_day-to-day_life,_how_often_are_you_threatened_or_harassed?": "harass_threatened",
        "In_your_day-to-day_life,_how_often_are_you_treated_with_less_courtesy_than_other_people?": "harass_courtesy",
        "In_your_day-to-day_life,_how_often_are_you_treated_with_less_respect_than_other_people?": "harass_respect",
        "In_your_day-to-day_life,_how_often_do_people_act_as_if_they_are_afraid_of_you?": "harass_afraid",
        "In_your_day-to-day_life,_how_often_do_people_act_as_if_he_or_she_think_you_are_dishonest?": "harass_dishonest",
        "In_your_day-to-day_life,_how_often_do_people_act_as_if_he_or_she_think_you_are_not_smart?": "harass_notsmart",
        "In_your_day-to-day_life,_how_often_do_people_act_as_if_theyre_better_than_you_are?": "harass_better",
        "In_your_day-to-day_life,_how_often_do_you_receive_poorer_service_than_other_people_at_restaurants_or_stores?": "harass_service",
    }
    mistrust_raw = {
        "How_often_are_you_treated_with_less_courtesy_than_other_people_when_you_go_to_a_doctors_office_or_other_health_care_provider?": "mistrust_courtesy",
        "How_often_are_you_treated_with_less_respect_than_other_people_when_you_go_to_a_doctors_office_or_other_health_care_provider?": "mistrust_respect",
        "How_often_do_you_feel_like_a_doctor_or_nurse_is_not_listening_to_what_you_were_saying._when_you_go_to_a_doctors_office_or_other_health_care_provider?": "mistrust_listening",
        "How_often_do_you_receive_poorer_service_than_others_when_you_go_to_a_doctors_office_or_other_health_care_provider?": "mistrust_service",
        "How_often_does_a_doctor_or_nurse_act_as_if_he_or_she_is_afraid_of_you_when_you_go_to_a_doctors_office_or_other_health_care_provider?": "mistrust_afraid",
        "How_often_does_a_doctor_or_nurse_act_as_if_he_or_she_is_better_than_you_when_you_go_to_a_doctors_office_or_other_health_care_provider?": "mistrust_better",
        "How_often_does_a_doctor_or_nurse_act_as_if_he_or_she_thinks_you_are_not_smart_when_you_go_to_a_doctors_office_or_other_health_care_provider?": "mistrust_notsmart",
    }
    stress_raw = {
        "In_the_last_month,_how_often_have_you_been_angered_because_of_things_that_were_outside_of_your_control?": "stress_angered",
        "In_the_last_month,_how_often_have_you_been_upset_because_of_something_that_happened_unexpectedly?": "stress_upset",
        "In_the_last_month,_how_often_have_you_felt_difficulties_were_piling_up_so_high_that_you_could_not_overcome_them?": "stress_difficulties",
        "In_the_last_month,_how_often_have_you_felt_nervous_and_stressed?": "stress_nervous",
        "In_the_last_month,_how_often_have_you_been_able_to_control_irritations_in_your_life?": "stress_irritations",
        "In_the_last_month,_how_often_have_you_felt_confident_about_your_ability_to_handle_your_personal_problems?": "stress_confident",
        "In_the_last_month,_how_often_have_you_felt_that_things_were_going_your_way?": "stress_things_going_way",
        "In_the_last_month,_how_often_have_you_felt_that_you_were_on_top_of_things?": "stress_on_top",
    }
    disorder_raw = {
        "How_much_you_agree_or_disagree_that_there_is_a_lot_of_crime_in_your_neighborhood?": "disorder_crime",
        "How_much_you_agree_or_disagree_that_there_is_too_much_drug_use_in_your_neighborhood?": "disorder_drugs",
        "How_much_you_agree_or_disagree_that_there_are_too_many_people_hanging_around_on_the_streets_near_your_home?": "disorder_loitering",
        "How_much_you_agree_or_disagree_that_your_neighborhood_is_noisy?": "disorder_noisy",
        "How_much_you_agree_or_disagree_that_there_is_too_much_alcohol_use_in_your_neighborhood?": "disorder_alcohol",
        "How_much_you_agree_or_disagree_that_there_is_a_lot_of_graffiti_in_your_neighborhood?": "disorder_graffiti",
        "How_much_you_agree_or_disagree_that_vandalism_is_common_in_your_neighborhood?": "disorder_vandalism",
    }

    def _encode_group(raw_to_concept: dict, mapping: dict) -> list[str]:
        cols = []
        for raw, concept in raw_to_concept.items():
            if raw in base.columns:
                _concept_missing_flag_from_raw(out, base[raw], concept)
                out[concept] = _encode_ordered_policy(base[raw], mapping)
            else:
                out[f"{concept}_missing"] = 1
                out[concept] = pd.NA
            cols.append(concept)
        return cols

    lonely_cols = _encode_group(loneliness_raw, lonely_map)
    harass_cols = _encode_group(harassment_raw, freq_map)
    mistrust_cols = _encode_group(mistrust_raw, freq_map)
    stress_cols = _encode_group(stress_raw, stress_map)
    disorder_cols = _encode_group(disorder_raw, agree_map)

    # composites (mean of available items)
    def _row_mean(df, cols):
        return df[cols].astype("Float64").mean(axis=1)

    out["loneliness_mean"] = _row_mean(out, lonely_cols)
    out["loneliness_mean_missing"] = out[[f"{c}_missing" for c in lonely_cols]].all(axis=1).astype("Int64")

    out["harassment_mean"] = _row_mean(out, harass_cols)
    out["harassment_mean_missing"] = out[[f"{c}_missing" for c in harass_cols]].all(axis=1).astype("Int64")

    out["mistrust_mean"] = _row_mean(out, mistrust_cols)
    out["mistrust_mean_missing"] = out[[f"{c}_missing" for c in mistrust_cols]].all(axis=1).astype("Int64")

    out["stress_mean"] = _row_mean(out, stress_cols)
    out["stress_mean_missing"] = out[[f"{c}_missing" for c in stress_cols]].all(axis=1).astype("Int64")

    out["neighborhood_disorder_mean"] = _row_mean(out, disorder_cols)
    out["neighborhood_disorder_mean_missing"] = out[[f"{c}_missing" for c in disorder_cols]].all(axis=1).astype("Int64")

    out = drop_response_behavior_flags(out)
    out = out.loc[:, ~out.columns.duplicated(keep="first")].copy()
    return out


# ==========================================================
# 6) Convenience wrappers to build ALL landmarks
# ==========================================================
def build_lifestyle_all_landmarks_compact(lifestyle_surveys, df_cutoff_baseline, df_cutoff_6m, df_cutoff_1y, df_cutoff_2y, df_cutoff_5y):
    return (
        prepare_lifestyle_compact_for_cutoff(lifestyle_surveys, df_cutoff_baseline),
        prepare_lifestyle_compact_for_cutoff(lifestyle_surveys, df_cutoff_6m),
        prepare_lifestyle_compact_for_cutoff(lifestyle_surveys, df_cutoff_1y),
        prepare_lifestyle_compact_for_cutoff(lifestyle_surveys, df_cutoff_2y),
        prepare_lifestyle_compact_for_cutoff(lifestyle_surveys, df_cutoff_5y),
    )


def build_sdoh_all_landmarks_compact(aou_social_health_surveys, df_cutoff_baseline, df_cutoff_6m, df_cutoff_1y, df_cutoff_2y, df_cutoff_5y):
    return (
        prepare_sdoh_compact_for_cutoff(aou_social_health_surveys, df_cutoff_baseline),
        prepare_sdoh_compact_for_cutoff(aou_social_health_surveys, df_cutoff_6m),
        prepare_sdoh_compact_for_cutoff(aou_social_health_surveys, df_cutoff_1y),
        prepare_sdoh_compact_for_cutoff(aou_social_health_surveys, df_cutoff_2y),
        prepare_sdoh_compact_for_cutoff(aou_social_health_surveys, df_cutoff_5y),
    )


In [ ]:
import pandas as pd
import numpy as np
import re

RELATIVE_TERMS = {
    "mother", "father", "parent",
    "sister", "brother", "sibling",
    "grandmother", "grandfather", "grandparent",
    "aunt", "uncle",
    "child", "son", "daughter",
    "cousin", "niece", "nephew",
    "relative", "other"
}
SELF_TERMS = {"yourself", "self", "me"}
NEGATIVE_TERMS = {"none_of_the_above", "none", "no", "no_one", "not_applicable"}
SKIP_TERMS = {"skip", "prefer_not_to_answer", "pna", "dont_know", "dont_know", "unknown"}

# Precompile regex for speed
_re_non_alnum = re.compile(r"[^a-z0-9_]+")
_re_multi_us = re.compile(r"_+")
_re_space = re.compile(r"\s+")

def _norm_str_series(s: pd.Series) -> pd.Series:
    """
    Normalize a string series quickly. Leaves NaN as NaN.
    """
    s = s.astype("string")
    s = s.str.lower().str.strip()
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(_re_space, " ", regex=True)
    s = s.str.replace(" ", "_", regex=False)
    s = s.str.replace(_re_non_alnum, "", regex=True)
    s = s.str.replace(_re_multi_us, "_", regex=True).str.strip("_")
    return s

def _shorten_fh_name(col: str) -> str:
    prefix = "including_yourself_who_in_your_family_has_had_"
    s = col.strip()
    s_low = s.lower()
    if s_low.startswith(prefix):
        s = s[len(prefix):]
    s = s.replace("?", "").replace("(", "_").replace(")", "")
    s = s.replace("/", "_")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_").lower()
    return "fh_" + s

def prepare_family_history_from_aou_fast(fh_df: pd.DataFrame) -> pd.DataFrame:
    if "person_id" not in fh_df.columns:
        raise ValueError("fh_df must contain 'person_id'")

    fh_cols = [
        c for c in fh_df.columns
        if c.lower().startswith("including_yourself_who_in_your_family_has_had")
    ]
    if not fh_cols:
        raise ValueError("No FH columns found with prefix 'Including_yourself_who_in_your_family_has_had'")

    out = pd.DataFrame({"person_id": fh_df["person_id"].astype("Int64")})

    # If any columns are list-like, coerce to comma-separated strings once (still faster than nested parsing)
    df = fh_df[["person_id"] + fh_cols].copy()

    for col in fh_cols:
        # Convert lists -> comma string; strings kept as-is
        # This is per-column but avoids per-token work.
        s = df[col]
        if s.apply(lambda x: isinstance(x, list)).any():
            s = s.apply(lambda x: ",".join(map(str, x)) if isinstance(x, list) else x)

        s = _norm_str_series(s)

        # Mark skip rows as NA
        # (any skip-like token anywhere in the response)
        skip_pat = r"(?:^|_)(?:%s)(?:_|$)" % "|".join(map(re.escape, sorted(SKIP_TERMS)))
        has_skip = s.str.contains(skip_pat, regex=True, na=False)

        # Remove self tokens (so FH only counts relatives)
        # Do this by replacing them with empty
        for st in SELF_TERMS:
            s = s.str.replace(rf"(?:^|_){re.escape(st)}(?:_|$)", "_", regex=True)

        s = s.str.replace(_re_multi_us, "_", regex=True).str.strip("_")

        # Negatives override to 0 (when not skip)
        neg_pat = r"(?:^|_)(?:%s)(?:_|$)" % "|".join(map(re.escape, sorted(NEGATIVE_TERMS)))
        has_neg = s.str.contains(neg_pat, regex=True, na=False)

        # Any relative token -> 1
        rel_pat = r"(?:^|_)(?:%s)(?:_|$)" % "|".join(map(re.escape, sorted(RELATIVE_TERMS)))
        has_rel = s.str.contains(rel_pat, regex=True, na=False)

        feat = _shorten_fh_name(col)

        encoded = np.where(has_skip, pd.NA, np.where(has_neg, 0, np.where(has_rel, 1, 0)))
        out[feat] = pd.Series(encoded, index=out.index, dtype="Int64")

    # If multiple rows per person exist, collapse fast with max (FH positive if ever positive)
    if out["person_id"].duplicated().any():
        out = out.groupby("person_id", as_index=False).max(numeric_only=True)

    # Optional cluster
    cluster_keywords = ["diabetes", "kidney", "ckd", "hypertension", "heart", "stroke", "tia", "cholesterol", "lipid"]
    cluster_cols = [c for c in out.columns if c != "person_id" and any(k in c for k in cluster_keywords)]
    out["fh_cardiometabolic_cluster"] = (out[cluster_cols].sum(axis=1) > 0).astype("Int64") if cluster_cols else pd.Series(pd.NA, index=out.index, dtype="Int64")

    return out


In [ ]:
DKD_FH_KEEP = [
    "person_id",

    # Core cardiometabolic
    "fh_type_2_diabetes",
    "fh_type_1_diabetes",
    "fh_prediabetes",
    "fh_high_blood_pressure_hypertension",
    "fh_high_cholesterol",
    "fh_obesity",

    # Kidney specific
    "fh_kidney_disease_without_dialysis",
    "fh_kidney_disease_with_dialysis",
    "fh_acute_kidney_disease_with_no_current_dialysis",  # optional
    "fh_other_kidney_condition_s",                       # optional
    "fh_kidney_stones",                                  # optional

    # Vascular and heart disease
    "fh_coronary_artery_coronary_heart_disease",
    "fh_a_heart_attack",
    "fh_congestive_heart_failure",
    "fh_peripheral_vascular_disease",
    "fh_a_stroke",
    "fh_transient_ischemic_attacks_tias_or_ministrokes",
    "fh_atrial_fibrillation_or_afib_or_atrial_flutter_or_aflutter",  # optional

    # Immune systemic (optional small add-on)
    "fh_systemic_lupus",
    "fh_rheumatoid_arthritis_ra",
    "fh_crohns_disease",
    "fh_ulcerative_colitis",
    "fh_celiac_disease",
]

def filter_fh_for_dkd(fh_out: pd.DataFrame, keep_cols=DKD_FH_KEEP) -> pd.DataFrame:
    keep = [c for c in keep_cols if c in fh_out.columns]
    df = fh_out[keep].copy()

    # rebuild a clean DKD cluster (instead of the huge generic cluster)
    cluster_cols = [c for c in df.columns if c != "person_id"]

    # if you want cluster to reflect only cardiometabolic + kidney (not immune)
    cardio_kidney_terms = [
        "diabetes", "prediabetes", "hypertension", "cholesterol", "obesity",
        "kidney", "heart", "coronary", "stroke", "tia", "vascular", "atrial_fibrillation"
    ]
    cluster_cols = [
        c for c in cluster_cols
        if any(t in c for t in cardio_kidney_terms)
    ]

    if cluster_cols:
        df["fh_dkd_cluster"] = (df[cluster_cols].sum(axis=1) > 0).astype("Int64")
    else:
        df["fh_dkd_cluster"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

    return df


In [ ]:
rename_map = {
    "fh_high_blood_pressure_hypertension": "fh_hypertension",
    "fh_coronary_artery_coronary_heart_disease": "fh_coronary_heart_disease",
    "fh_congestive_heart_failure": "fh_heart_failure",
    "fh_peripheral_vascular_disease": "fh_pvd",
    "fh_transient_ischemic_attacks_tias_or_ministrokes": "fh_tia",
    "fh_atrial_fibrillation_or_afib_or_atrial_flutter_or_aflutter": "fh_atrial_fibrillation",
    "fh_rheumatoid_arthritis_ra": "fh_rheumatoid_arthritis",
    "fh_crohns_disease": "fh_crohns",
    "fh_celiac_disease": "fh_celiac",
    "fh_acute_kidney_disease_with_no_current_dialysis": "fh_aki",
    "fh_kidney_disease_without_dialysis": "fh_ckd_no_dialysis",
    "fh_kidney_disease_with_dialysis": "fh_ckd_dialysis",
    "fh_other_kidney_condition_s": "fh_other_kidney_conditions",
}


In [ ]:
def prepare_family_history_for_cutoff(
    fh_df,
    cutoff_df,
    date_col="personal_family_history_survey_date",
):
    """
    Build family-history features using only a survey completed
    strictly before the participant's landmark.
    """
    if date_col not in fh_df.columns:
        raise KeyError(
            f"Family-history survey date column '{date_col}' is required "
            "for landmark-safe feature construction."
        )

    selected, _ = select_latest_pre_cutoff_one_row(
        survey_df=fh_df,
        cutoff_df=cutoff_df,
        id_col="person_id",
        cutoff_col="cutoff_date",
        date_col=date_col,
        max_lookback_days=None,
    )

    if selected.empty:
        return cutoff_df[["person_id"]].drop_duplicates().copy()

    fh_out = prepare_family_history_from_aou_fast(
        selected
    )

    fh_dkd = filter_fh_for_dkd(
        fh_out
    )

    fh_dkd = fh_dkd.rename(
        columns={
            k: v
            for k, v in rename_map.items()
            if k in fh_dkd.columns
        }
    )

    ensure_person_level(
        fh_dkd,
        "family_history",
    )

    return fh_dkd


In [ ]:
survey_blocks = {}

for name, cohort in landmark_cohorts.items():
    print(f"\nBuilding survey features: {name}")

    # Basics survey: select the latest response strictly before
    # the participant's landmark, then apply the compact encoding policy.
    basic_selected, _ = select_latest_pre_cutoff_one_row(
        survey_df=aou_basic_surveys,
        cutoff_df=cohort,
        id_col="person_id",
        cutoff_col="cutoff_date",
        date_col="basics_survey_date",
        max_lookback_days=None,
    )

    demographics = build_demographics_basic_static(
        aou_basic_surveys=basic_selected,
        cohort_df=cohort,
        survey_date_col="basics_survey_date",
    )

    lifestyle = prepare_lifestyle_compact_for_cutoff(
        lifestyle_surveys,
        cohort,
    )

    sdoh = prepare_sdoh_compact_for_cutoff(
        aou_social_health_surveys,
        cohort,
    )

    family_history = prepare_family_history_for_cutoff(
        fh_df,
        cohort,
    )

    backbone = (
        cohort[["person_id"]]
        .drop_duplicates()
        .copy()
    )

    block = (
        backbone
        .merge(
            demographics,
            on="person_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            lifestyle,
            on="person_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            sdoh,
            on="person_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            family_history,
            on="person_id",
            how="left",
            validate="one_to_one",
        )
    )

    block = block.loc[
        :,
        ~block.columns.duplicated(),
    ].copy()

    ensure_person_level(
        block,
        f"surveys_{name}",
    )

    survey_blocks[name] = block

    save_gcs_parquet(
        block,
        f"{CURATED_DIR}/surveys_{name}.parquet",
    )


## Polygenic Risk Scores

Available PRS files are loaded from the migrated DKD and stroke PRS directories. PRS values are static across landmarks and are merged by participant ID.


In [ ]:
import pandas as pd

def compute_final_score(df, name="score"):
    """
    Compute a final PRS score per person.
    Automatically handles variant naming conventions:
    - 'sample_id' or 's' as ID column
    - Normalizes by N_variants if available and variable
    """

    df = df.copy()

    # --- Handle sample ID column ---
    if "sample_id" not in df.columns:
        if "s" in df.columns:
            df = df.rename(columns={"s": "sample_id"})
        elif "IID" in df.columns:
            df = df.rename(columns={"IID": "sample_id"})
        else:
            raise ValueError("No sample identifier column found (expected 'sample_id' or 's').")

    # --- Compute final score ---
    if {"sum_weights", "N_variants"}.issubset(df.columns):
        df["N_variants"] = df["N_variants"].astype(int)
        normalize = df["N_variants"].nunique() > 1

        if normalize:
            print(f"Normalizing by N_variants (range {df['N_variants'].min()}–{df['N_variants'].max()}).")
            df[name] = df["sum_weights"] / df["N_variants"]
        else:
            print(f"N_variants constant ({df['N_variants'].iloc[0]}). Using raw sum_weights.")
            df[name] = df["sum_weights"]

    elif "sum_weights" in df.columns:
        print("⚠️ 'N_variants' missing. Using raw sum_weights as final score.")
        df[name] = df["sum_weights"]

    elif "score" in df.columns:
        print("⚠️ Using existing 'score' column directly.")
        df[name] = df["score"]

    else:
        raise ValueError("No valid columns found to compute PRS ('sum_weights' or 'score').")

    return df.rename(columns={"sample_id": "person_id"})[["person_id", name]]


In [ ]:
# --- Directories to search ---
prs_dirs = [
    "aou_stroke/stroke_prs",
    "aou_dkd/dkd_prs"
]

# --- PGS IDs and names ---
prs_list = {
    # aou_stroke/stroke_prs/
    "PGS002724": "PGS002724_Stroke_PRS",
    "PGS000039": "PGS000039_Stroke_PRS",
    "PGS005313": "PGS005313_Afib_PRS",
    "PGS003966": "PGS003966_SBP_PRS",
    "PGS003962": "PGS003962_DBP_PRS",
    "PGS004236": "PGS004236_Hypertension_PRS",
    "PGS004888": "PGS004888_CAD_PRS",
    "PGS000888": "PGS000888_LDL_PRS",
    "PGS005198": "PGS005198_BMI_PRS",
    "PGS002126": "PGS002126_Smoking_PRS",
    
    # aou_dkd/dkd_prs/
    "PGS004889": "PGS004889_CKD_PRS",
    "PGS000036": "PGS000036_T2D_PRS",
    "PGS000682": "PGS000682_eGFR_PRS",
    "PGS000884": "PGS000884_eGFR_PRS",
    "PGS003309": "PGS003309_eGFR_CysC_PRS",
    "PGS000305": "PGS000305_FG_PRS",
    "PGS003469": "PGS003469_HOMAB_PRS",
    "PGS003470": "PGS003470_HOMAIR_PRS",
    "PGS000304": "PGS000304_HbA1c_PRS",
    "PGS000822": "PGS000822_UACR_PRS",
    "PGS004896": "PGS004896_Microalbuminuria_PRS",
    "PGS000861": "PGS000861_UACR_PRS"
}

# --- Read and merge ---
fs = gcsfs.GCSFileSystem()
prs_merged_df = None

for pgs_id, col_name in prs_list.items():
    print(f"\nProcessing {pgs_id} → {col_name}")

    found_path = None

    # search stroke and dkd folders
    for base_dir in prs_dirs:
        for fname in [f"{pgs_id}_final_scores.csv", f"{pgs_id}_scores.csv"]:
            path = f"{bucket}/{base_dir}/{pgs_id}/score/score/{fname}"
            if fs.exists(path):
                found_path = path
                break
        if found_path:
            break

    if not found_path:
        print(f"⚠️  File not found for {pgs_id} in any directory, skipping.")
        continue

    print(f"Reading {found_path}")
    with fs.open(found_path, "rb") as f:
        df = pd.read_csv(f)

    df_final = compute_final_score(df, name=col_name)

    if prs_merged_df is None:
        prs_merged_df = df_final
    else:
        prs_merged_df = prs_merged_df.merge(df_final, on="person_id", how="outer")

print("\n✅ All available PRS files merged successfully.")
print("Final merged shape:", prs_merged_df.shape)


In [ ]:
if prs_merged_df is None:
    prs_merged_df = pd.DataFrame(
        columns=["person_id"]
    )

prs_merged_df["person_id"] = pd.to_numeric(
    prs_merged_df["person_id"],
    errors="coerce",
).astype("Int64")

prs_merged_df = (
    prs_merged_df
    .dropna(subset=["person_id"])
    .drop_duplicates("person_id")
)

save_gcs_parquet(
    prs_merged_df,
    f"{CURATED_DIR}/polygenic_risk_scores.parquet",
)


## Temporal Features

Age at T2D diagnosis and elapsed time from T2D diagnosis to the prediction landmark are calculated from the curated demographic and index-date data.


In [ ]:
def build_temporal_features(
    cutoff_df,
    demographics_df,
):
    """Build age-at-T2D and T2D-duration features."""
    dob = (
        demographics_df[
            ["person_id", "date_of_birth"]
        ]
        .drop_duplicates("person_id")
        .copy()
    )

    df = cutoff_df[
        [
            "person_id",
            "t2d_first_evidence_date",
            "cutoff_date",
        ]
    ].merge(
        dob,
        on="person_id",
        how="left",
        validate="one_to_one",
    )

    for col in [
        "t2d_first_evidence_date",
        "cutoff_date",
        "date_of_birth",
    ]:
        df[col] = parse_datetime(df[col])

    df["age_at_t2d"] = (
        df["t2d_first_evidence_date"]
        - df["date_of_birth"]
    ).dt.days / 365.25

    df["t2d_duration"] = (
        df["cutoff_date"]
        - df["t2d_first_evidence_date"]
    ).dt.days / 365.25

    df.loc[
        df["t2d_duration"] < 0,
        "t2d_duration",
    ] = np.nan

    return df[
        [
            "person_id",
            "age_at_t2d",
            "t2d_duration",
        ]
    ]


temporal_blocks = {
    name: build_temporal_features(
        cohort,
        participant_demographics,
    )
    for name, cohort in landmark_cohorts.items()
}


# 6. Modeling-Ready Landmark Matrices

Each final matrix begins with the validated landmark cohort and left-joins the feature blocks constructed for that landmark.

The merge is row-preserving: no feature block is allowed to add, remove, or duplicate participants.


In [ ]:
def merge_feature_blocks(
    cohort,
    blocks,
    label,
):
    """Left-join feature blocks while preserving the landmark row set."""
    base_cols = [
        "person_id",
        "dkd_binary",
        "t2d_first_evidence_date",
        "cutoff_date",
        "kidney_first_evidence_date",
        "ehr_end_date",
        "landmark_name",
        "landmark_days",
    ]

    out = cohort[base_cols].copy()
    ensure_person_level(out, f"{label}_base")

    expected_ids = set(
        out["person_id"].astype(int)
    )

    for block_name, block in blocks:
        if block is None or len(block) == 0:
            print(
                f"{label}: skipping empty block "
                f"{block_name}"
            )
            continue

        x = block.copy()
        x["person_id"] = pd.to_numeric(
            x["person_id"],
            errors="coerce",
        ).astype("Int64")

        x = x.dropna(
            subset=["person_id"]
        )

        ensure_person_level(
            x,
            f"{label}_{block_name}",
        )

        # Avoid silently duplicating feature names.
        overlap = (
            set(out.columns)
            & set(x.columns)
        ) - {"person_id"}

        if overlap:
            raise ValueError(
                f"{label}/{block_name}: overlapping "
                f"feature columns: {sorted(overlap)[:20]}"
            )

        out = out.merge(
            x,
            on="person_id",
            how="left",
            validate="one_to_one",
        )

        assert len(out) == len(cohort)

    assert set(
        out["person_id"].astype(int)
    ) == expected_ids

    return out


In [ ]:
feature_matrices = {}

for name, cohort in landmark_cohorts.items():
    blocks = [
        ("temporal", temporal_blocks[name]),
        ("comorbidities", comorbidity_blocks[name]),
        ("core_labs", core_lab_blocks[name]),
        ("additional_labs", additional_lab_blocks[name]),
        ("body", body_blocks[name]),
        ("medications", medication_blocks[name]),
        ("surveys", survey_blocks[name]),
        ("prs", prs_merged_df),
    ]

    matrix = merge_feature_blocks(
        cohort,
        blocks,
        label=name,
    )

    feature_matrices[name] = matrix

    print(
        f"{name}: final matrix "
        f"{matrix.shape[0]:,} x {matrix.shape[1]:,}"
    )


## Final Quality Control and Save

The final matrices are checked for participant uniqueness, outcome consistency, and absence of post-landmark DKD in the risk set. Pickle files are saved for downstream modeling, with Parquet copies for portable inspection.


In [ ]:
FINAL_MATRIX_PATHS = {}

for name, matrix in feature_matrices.items():
    cohort = landmark_cohorts[name]

    assert matrix["person_id"].is_unique
    assert len(matrix) == len(cohort)

    assert (
        matrix["dkd_binary"]
        .value_counts(dropna=False)
        .sum()
        == len(matrix)
    )

    # Every modeled case satisfies both temporal rules:
    # incident DKD begins no earlier than day 180 and occurs
    # strictly after the prediction landmark.
    case_mask = matrix["dkd_binary"].eq(1)

    assert (
        (
            matrix.loc[
                case_mask,
                "kidney_first_evidence_date",
            ]
            - matrix.loc[
                case_mask,
                "t2d_first_evidence_date",
            ]
        ).dt.days
        >= INCIDENT_LAG_DAYS
    ).all()

    assert (
        matrix.loc[
            case_mask,
            "kidney_first_evidence_date",
        ]
        > matrix.loc[
            case_mask,
            "cutoff_date",
        ]
    ).all()

    assert matrix.loc[
        ~case_mask,
        "kidney_first_evidence_date",
    ].isna().all()

    pickle_path = (
        f"{CURATED_DIR}/"
        f"dkd_risk_data_compact_surveys_{name}.pkl"
    )

    parquet_path = (
        f"{CURATED_DIR}/"
        f"dkd_risk_data_compact_surveys_{name}.parquet"
    )

    save_gcs_pickle(
        matrix,
        pickle_path,
    )

    save_gcs_parquet(
        matrix,
        parquet_path,
    )

    FINAL_MATRIX_PATHS[name] = {
        "pickle": pickle_path,
        "parquet": parquet_path,
    }


In [ ]:
final_summary = pd.DataFrame(
    [
        {
            "landmark": name,
            "n": len(matrix),
            "cases": int(
                matrix["dkd_binary"].sum()
            ),
            "controls": int(
                (matrix["dkd_binary"] == 0).sum()
            ),
            "features_excluding_id_outcome": (
                matrix.shape[1] - 2
            ),
            "pickle_path": FINAL_MATRIX_PATHS[name]["pickle"],
            "parquet_path": FINAL_MATRIX_PATHS[name]["parquet"],
        }
        for name, matrix in feature_matrices.items()
    ]
)

save_gcs_csv(
    final_summary,
    f"{CURATED_DIR}/final_feature_matrix_summary.csv",
)

final_summary


In [ ]:
# ============================================================
# Case overlap across landmarks
# ============================================================

case_sets = {
    "30d": set(
        df_cutoff_baseline.loc[
            df_cutoff_baseline["dkd_binary"].eq(1),
            "person_id",
        ].astype(int)
    ),
    "6m": set(
        df_cutoff_6m.loc[
            df_cutoff_6m["dkd_binary"].eq(1),
            "person_id",
        ].astype(int)
    ),
    "1y": set(
        df_cutoff_1y.loc[
            df_cutoff_1y["dkd_binary"].eq(1),
            "person_id",
        ].astype(int)
    ),
    "2y": set(
        df_cutoff_2y.loc[
            df_cutoff_2y["dkd_binary"].eq(1),
            "person_id",
        ].astype(int)
    ),
}

landmarks = list(case_sets)


# ------------------------------------------------------------
# 1. Case counts
# ------------------------------------------------------------

print("=== Case counts by landmark ===")

for landmark in landmarks:
    print(
        f"{landmark:>3}: "
        f"{len(case_sets[landmark]):,}"
    )


# ------------------------------------------------------------
# 2. Pairwise intersection counts
# ------------------------------------------------------------

pairwise_overlap = pd.DataFrame(
    index=landmarks,
    columns=landmarks,
    dtype=int,
)

for row_landmark in landmarks:
    for col_landmark in landmarks:
        pairwise_overlap.loc[
            row_landmark,
            col_landmark,
        ] = len(
            case_sets[row_landmark]
            & case_sets[col_landmark]
        )

pairwise_overlap = pairwise_overlap.astype(int)

print("\n=== Pairwise case intersections ===")
display(pairwise_overlap)


# ------------------------------------------------------------
# 3. Participant-level membership table
# ------------------------------------------------------------

all_case_ids = set().union(
    *case_sets.values()
)

case_membership = pd.DataFrame(
    {
        "person_id": sorted(all_case_ids)
    }
)

for landmark in landmarks:
    case_membership[
        f"case_{landmark}"
    ] = (
        case_membership["person_id"]
        .isin(case_sets[landmark])
        .astype(int)
    )

case_membership["n_landmarks_case"] = (
    case_membership[
        [f"case_{x}" for x in landmarks]
    ]
    .sum(axis=1)
)

case_membership.head()

In [ ]:
# Case present at 30d but not at 6m
case_30d_only = (
    case_sets["30d"]
    - case_sets["6m"]
)

print(case_30d_only)

df_cutoff_baseline.loc[
    df_cutoff_baseline["person_id"]
    .astype(int)
    .isin(case_30d_only),
    [
        "person_id",
        "t2d_first_evidence_date",
        "kidney_first_evidence_date",
        "days_kidney_after_t2d",
        "dkd_binary",
    ],
]

In [ ]:
ids_30d = set(
    df_cutoff_baseline["person_id"].astype(int)
)

ids_6m = set(
    df_cutoff_6m["person_id"].astype(int)
)

print("30d N:", len(ids_30d))
print("6m N:", len(ids_6m))
print("30d only:", len(ids_30d - ids_6m))
print("6m only:", len(ids_6m - ids_30d))
print("Shared:", len(ids_30d & ids_6m))

In [ ]:
# ============================================================
# Harmonize 30d and 6m cohorts
# Same participants and outcomes; different feature windows
# ============================================================

common_early_ids = set(
    df_cutoff_6m["person_id"].astype(int)
)

df_cutoff_baseline = (
    df_cutoff_baseline.loc[
        df_cutoff_baseline["person_id"]
        .astype(int)
        .isin(common_early_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

# Verify identical participant membership.
assert set(
    df_cutoff_baseline["person_id"].astype(int)
) == set(
    df_cutoff_6m["person_id"].astype(int)
)

# Verify identical outcomes.
early_outcome_check = (
    df_cutoff_baseline[
        ["person_id", "dkd_binary"]
    ]
    .merge(
        df_cutoff_6m[
            ["person_id", "dkd_binary"]
        ],
        on="person_id",
        suffixes=("_30d", "_6m"),
        validate="one_to_one",
    )
)

assert (
    early_outcome_check["dkd_binary_30d"]
    == early_outcome_check["dkd_binary_6m"]
).all()

print(
    "30d and 6m cohorts are identical:",
    f"N={len(df_cutoff_baseline):,},",
    f"cases={df_cutoff_baseline['dkd_binary'].sum():,},",
    f"controls={(df_cutoff_baseline['dkd_binary'] == 0).sum():,}",
)

In [ ]:
# ============================================================
# Save final modeling-ready landmark matrices
# ============================================================

FINAL_DATE = "20260822"

# Harmonize the 30-day feature matrix to the finalized 6-month cohort
early_ids = set(
    df_cutoff_6m["person_id"].astype(int)
)

feature_matrices["30d"] = (
    feature_matrices["30d"]
    .loc[
        feature_matrices["30d"]["person_id"]
        .astype(int)
        .isin(early_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

# Confirm identical 30d / 6m participants
assert set(
    feature_matrices["30d"]["person_id"].astype(int)
) == set(
    feature_matrices["6m"]["person_id"].astype(int)
)

# Confirm identical 30d / 6m outcomes
early_check = (
    feature_matrices["30d"][
        ["person_id", "dkd_binary"]
    ]
    .merge(
        feature_matrices["6m"][
            ["person_id", "dkd_binary"]
        ],
        on="person_id",
        suffixes=("_30d", "_6m"),
        validate="one_to_one",
    )
)

assert (
    early_check["dkd_binary_30d"]
    == early_check["dkd_binary_6m"]
).all()


FINAL_MODEL_PATHS = {}

for landmark in ["30d", "6m", "1y", "2y"]:

    df = feature_matrices[landmark]

    pickle_path = (
        f"{CURATED_DIR}/"
        f"DKD_Risk_Modeling_Data_Landmark_"
        f"{landmark}_{FINAL_DATE}.pkl"
    )

    parquet_path = (
        f"{CURATED_DIR}/"
        f"DKD_Risk_Modeling_Data_Landmark_"
        f"{landmark}_{FINAL_DATE}.parquet"
    )

    save_gcs_parquet(
        df,
        parquet_path,
    )

    with fs.open(pickle_path, "wb") as f:
        df.to_pickle(f)

    FINAL_MODEL_PATHS[landmark] = {
        "pickle": pickle_path,
        "parquet": parquet_path,
    }

    print(
        f"{landmark:>3}: "
        f"N={len(df):,}, "
        f"cases={int(df['dkd_binary'].sum()):,}, "
        f"controls={int((df['dkd_binary'] == 0).sum()):,}, "
        f"columns={df.shape[1]:,}"
    )

print("\nFinal modeling-ready landmark matrices saved.")

In [ ]:
assert set(
    df_cutoff_baseline["person_id"].astype(int)
) == set(
    df_cutoff_6m["person_id"].astype(int)
)

assert (
    df_cutoff_baseline[
        ["person_id", "dkd_binary"]
    ]
    .merge(
        df_cutoff_6m[
            ["person_id", "dkd_binary"]
        ],
        on="person_id",
        suffixes=("_30d", "_6m"),
        validate="one_to_one",
    )
    .eval("dkd_binary_30d == dkd_binary_6m")
    .all()
)

print("30d and 6m cohorts have identical participants and outcomes.")

In [ ]:
for landmark in ["30d", "6m", "1y", "2y"]:

    pkl_path = (
        f"{CURATED_DIR}/"
        f"DKD_Risk_Modeling_Data_Landmark_"
        f"{landmark}_20260822.pkl"
    )

    parquet_path = (
        f"{CURATED_DIR}/"
        f"DKD_Risk_Modeling_Data_Landmark_"
        f"{landmark}_20260822.parquet"
    )

    print(
        f"{landmark}: "
        f"PKL={fs.exists(pkl_path)} | "
        f"Parquet={fs.exists(parquet_path)}"
    )